# DiT Fig.2 Generalization Sweep: Results Check

Clean result notebook for the DiT Fig.2-style memorization/generalization sweep.
It compares DiT depth variants (`DiT-L8`, `DiT-L12/base`, `DiT-L16`) against the existing UNet baselines using the same PCA and SSCD nearest-neighbor diagnostics.

## tl;dr

- The DiT jobs use the same Fig.2-style protocol as the UNet sweep: fixed update budget, no augmentation, generated samples evaluated by PCA/SSCD nearest-neighbor similarity.
- Treat `DiT-L8`, `DiT-L12/base`, and `DiT-L16` as separate architectures. Do not collapse them into one “DiT” curve.
- The first thing to check is whether the DiT transition shifts with depth, then whether DiT behaves like an equal-parameter UNet or follows a different architecture-specific curve.

## Nick review checklist

This notebook now answers the three review questions directly:

1. **Nearest training examples:** generated DiT maps are shown beside their nearest slice from the **exact training subset used by that model**, with a difference map and pixel-space similarity values.
2. **Black physical-statistics reference:** every one-point and $P(k)$ black curve is rebuilt from that model's configured training subset. The notebook checks the configured slice count against $N_{2D}$ and reports any plotting cap.
3. **In-distribution check:** an **SSCD Fréchet distance** compares generated maps with heldout real maps and is divided by a **real-vs-real split baseline**. A ratio near one means the generated set is no farther from heldout data than two finite real subsets are from each other. A much larger ratio flags novel but potentially invalid samples.

The third check is FID-style but uses SSCD features instead of literal ImageNet Inception features. Literal ImageNet Inception is a weak representation for single-channel HI fields; the calculation and finite-sample baseline are otherwise the same Fréchet distribution comparison.


## Setup


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from matplotlib.lines import Line2D
from IPython.display import Image, Markdown, display


def parse_csv_env(name: str, default: str) -> list[str]:
    return [x.strip() for x in os.environ.get(name, default).split(',') if x.strip()]


def find_project_dir() -> Path:
    env = os.environ.get('DIFFUSION_PROJECT_DIR')
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'notebooks').exists():
            return candidate
    return here


PROJECT_DIR = find_project_dir()
SWEEP_NAME = 'nf_generalize_fig2_dit'
RESULTS_DIR = PROJECT_DIR / 'results' / SWEEP_NAME
TABLE_DIR = RESULTS_DIR / 'tables'
QUICKCHECK_DIR = RESULTS_DIR / 'quickcheck'
SAMPLE_DIR = RESULTS_DIR / 'samples'
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
SAMPLE_LABEL = os.environ.get('SAMPLE_LABEL', 'dpm50')
SEED = int(os.environ.get('SEED', '123'))

DIT_ARCH_ORDER = ['dit_l8', 'dit_base', 'dit_l16']
DIT_ARCH_LABELS = {
    'dit_l8': 'DiT-L8',
    'dit_base': 'DiT-L12 / base',
    'dit': 'DiT-L12 / base',
    'dit_l16': 'DiT-L16',
}
DIT_ARCH_COLORS = {
    'dit_l8': '#009E73',
    'dit_base': '#0072B2',
    'dit': '#0072B2',
    'dit_l16': '#CC79A7',
}
DIT_ARCH_MARKERS = {'dit_l8': 'P', 'dit_base': 'D', 'dit': 'D', 'dit_l16': 'X'}
UNET_ARCH_ORDER = ['u64', 'u128', 'u256']
UNET_ARCH_LABELS = {'u64': 'UNet-64', 'u128': 'UNet-128', 'u256': 'UNet-256'}
UNET_ARCH_COLORS = {'u64': '#009E73', 'u128': '#D55E00', 'u256': '#0072B2'}
UNET_ARCH_MARKERS = {'u64': '^', 'u128': 'o', 'u256': 's'}

DETAIL_ARCH = os.environ.get('DIT_DETAIL_ARCH', 'dit_base')
IMAGE_ARCH = os.environ.get('DIT_IMAGE_ARCH', DETAIL_ARCH)
LOSS_ARCHES = parse_csv_env('DIT_LOSS_ARCHES', 'dit_l8,dit_base,dit_l16')
LOSS_TAGS = parse_csv_env('DIT_LOSS_TAGS', 'd2p06,d2p10,d2p15')
DETAIL_TAGS = parse_csv_env('DIT_DETAIL_TAGS', 'd2p06,d2p08,d2p10,d2p12,d2p15')
IMAGE_TAGS = parse_csv_env('DIT_IMAGE_TAGS', 'd2p06,d2p08,d2p10,d2p12,d2p15')

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

print('PROJECT_DIR =', PROJECT_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
print('SAMPLE_LABEL =', SAMPLE_LABEL)
print('SEED =', SEED)
print('DETAIL_ARCH =', DETAIL_ARCH, 'IMAGE_ARCH =', IMAGE_ARCH)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

try:
    from simdiff_eval.io import (
        as_nchw, configured_training_reference_info, load_real_from_config,
        load_real_reference_from_config,
    )
    from simdiff_eval.metrics import batch_power_spectra, field_histogram, frechet_feature_distance, nearest_training_matches, real_split_frechet_baseline
    SIMDIFF_EVAL_AVAILABLE = True
except Exception as exc:
    SIMDIFF_EVAL_AVAILABLE = False
    SIMDIFF_EVAL_ERROR = repr(exc)
    print('simdiff_eval unavailable; image/P(k) diagnostics will be skipped:', SIMDIFF_EVAL_ERROR)

PK_NBINS = int(os.environ.get('DIT_PK_NBINS', '30'))
MAX_GENERATED = int(os.environ.get('DIT_MAX_GENERATED', '512'))
MAX_REAL_REFERENCE_SLICES = int(os.environ.get('DIT_MAX_REAL_REFERENCE_SLICES', '2048'))

## Context & Methods

These runs use Hugging Face diffusers `DiTTransformer2DModel` with a single null class label (`class_labels=0`) because the DiT adaLN path requires labels even for unconditional training. The label is constant, so this is still an unconditional Fig.2-style sweep.

The depth sweep is:

- `DiT-L8`: 8 transformer blocks, width 768.
- `DiT-L12/base`: the original DiT-base run, 12 transformer blocks, width 768.
- `DiT-L16`: 16 transformer blocks, width 768.

Each model is trained at the same 2D training-set sizes as the UNet Fig.2 sweep, then sampled with DPM-Solver 50 steps. The diagnostics below ask whether deeper DiTs need more data before generated fields stop looking like nearest training slices.

## Data Audit


In [ ]:
def read_json(path: Path) -> Any | None:
    if not path.exists():
        return None
    with path.open() as f:
        return json.load(f)


def rel(path: Path | str | None) -> str:
    if path is None:
        return ''
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        return str(path)


def dataset_tag_from_name(name: str) -> str | None:
    m = re.search(r'd2p(\d+)', str(name))
    if not m:
        return None
    return 'd2p' + m.group(1)


def dataset_size_from_tag(tag: str | None) -> int | None:
    if not tag:
        return None
    m = re.match(r'd2p(\d+)', tag)
    if not m:
        return None
    return 2 ** int(m.group(1))


def arch_from_run_name(name: str) -> str:
    text = str(name)
    for arch in ['dit_l16', 'dit_l8', 'dit_base', 'u256', 'u128', 'u64']:
        if arch in text:
            return arch
    if re.search(r'\bdit\b', text):
        return 'dit_base'
    return 'unknown'


def arch_label(raw: Any) -> str:
    text = str(raw)
    if text in DIT_ARCH_LABELS:
        return DIT_ARCH_LABELS[text]
    if text in UNET_ARCH_LABELS:
        return UNET_ARCH_LABELS[text]
    return text


def ensure_arch_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    if 'arch' not in out.columns:
        name_col = 'run_name' if 'run_name' in out.columns else ('name' if 'name' in out.columns else None)
        out['arch'] = out[name_col].map(arch_from_run_name) if name_col else 'unknown'
    else:
        out['arch'] = out['arch'].astype(str).replace({'dit': 'dit_base'})
    if 'arch_label' not in out.columns:
        out['arch_label'] = out['arch'].map(arch_label)
    return out


def dataset_size_label(n: int | float) -> str:
    n = int(n)
    log2n = np.log2(n)
    if np.isfinite(log2n) and abs(log2n - round(log2n)) < 1e-9:
        return rf'$2^{{{int(round(log2n))}}}$'
    return f'{n:,}'


def sample_path_for(row: pd.Series) -> Path:
    raw = str(row.get('sample_path', '') or '')
    if raw:
        raw = raw.format(seed=SEED, sample_label=SAMPLE_LABEL)
        path = Path(raw)
        return path if path.is_absolute() else PROJECT_DIR / path
    run_name = row.get('run_name') or row.get('name')
    arch = row.get('arch') or arch_from_run_name(str(run_name))
    tag = row.get('dataset_tag') or dataset_tag_from_name(str(run_name))
    if tag is None:
        return SAMPLE_DIR / f'unknown_seed{SEED}_{SAMPLE_LABEL}.npz'
    return SAMPLE_DIR / f'nf_fig2_{arch}_{tag}_noaug_200k_seed{SEED}_{SAMPLE_LABEL}.npz'


manifest_obj = read_json(MANIFEST_PATH)
if manifest_obj is None:
    display(Markdown(f'**Missing manifest:** `{rel(MANIFEST_PATH)}`'))
    manifest_df = pd.DataFrame()
else:
    if isinstance(manifest_obj, dict):
        rows = manifest_obj.get('runs', [])
    elif isinstance(manifest_obj, list):
        rows = manifest_obj
    else:
        rows = []
    manifest_df = pd.DataFrame(rows)
    if 'run_name' not in manifest_df.columns and 'name' in manifest_df.columns:
        manifest_df['run_name'] = manifest_df['name']
    manifest_df = ensure_arch_columns(manifest_df)
    if 'dataset_tag' not in manifest_df.columns:
        manifest_df['dataset_tag'] = manifest_df['run_name'].map(dataset_tag_from_name)
    if 'dataset_size' not in manifest_df.columns:
        manifest_df['dataset_size'] = manifest_df['dataset_tag'].map(dataset_size_from_tag)
    manifest_df['sample_path_resolved'] = manifest_df.apply(sample_path_for, axis=1)
    manifest_df['sample_exists'] = manifest_df['sample_path_resolved'].map(Path.exists)
    manifest_df['sample_size_mb'] = manifest_df['sample_path_resolved'].map(lambda p: p.stat().st_size / 1024**2 if p.exists() else np.nan)

    show_cols = [c for c in [
        'arch', 'arch_label', 'run_name', 'dataset_tag', 'dataset_size', 'sample_exists', 'sample_size_mb',
        'config_path', 'output_dir', 'sample_path_resolved'
    ] if c in manifest_df.columns]
    display(manifest_df.sort_values(['arch', 'dataset_size'])[show_cols])
    print(f"sample files present: {manifest_df['sample_exists'].sum()} / {len(manifest_df)}")

expected_tables = [
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_metrics.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_mode_norms.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_histograms.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_metrics.csv',
]
expected_figures = [
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_paper_style_gl_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_paper_style_gl_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_similarity_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_copy_fraction_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_copy_fraction_curves.png',
]

audit = pd.DataFrame({
    'path': [rel(p) for p in expected_tables + expected_figures],
    'kind': ['table'] * len(expected_tables) + ['figure'] * len(expected_figures),
    'exists': [p.exists() for p in expected_tables + expected_figures],
    'size_mb': [p.stat().st_size / 1024**2 if p.exists() else np.nan for p in expected_tables + expected_figures],
})
display(audit)

## Load Metrics


In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        display(Markdown(f'**Missing table:** `{rel(path)}`'))
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'loaded {rel(path)}: {len(df)} rows, {len(df.columns)} columns')
    return df


def add_generalization_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = ensure_arch_columns(df)
    for q in ['q50', 'q68', 'q90', 'q95', 'q99']:
        copy_col = f'gen_copy_fraction_{q}'
        gl_col = f'gen_gl_{q}'
        if gl_col not in df.columns and copy_col in df.columns:
            df[gl_col] = 1.0 - df[copy_col]
    # Some older tables used shorter names.
    for q in ['q90', 'q95', 'q99']:
        if f'gen_gl_{q}' not in df.columns and f'copy_fraction_{q}' in df.columns:
            df[f'gen_gl_{q}'] = 1.0 - df[f'copy_fraction_{q}']
    if 'dataset_tag' not in df.columns:
        name_col = 'run_name' if 'run_name' in df.columns else df.columns[0]
        df['dataset_tag'] = df[name_col].map(dataset_tag_from_name)
    if 'dataset_size' not in df.columns:
        df['dataset_size'] = df['dataset_tag'].map(dataset_size_from_tag)
    return df


pca_metrics = add_generalization_columns(read_csv_if_exists(TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_metrics.csv'))
sscd_metrics = add_generalization_columns(read_csv_if_exists(TABLE_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_metrics.csv'))

for feature_name, df in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
    display(Markdown(f'### {feature_name} metrics'))
    if df.empty:
        continue
    print('architectures:', sorted(df['arch'].dropna().unique()) if 'arch' in df.columns else 'missing')
    print('generalization columns:', [c for c in df.columns if c.startswith('gen_gl')])
    preferred = [
        'arch_label', 'run_name', 'dataset_tag', 'dataset_size', 'n_generated', 'n_train',
        'gen_gl_q90', 'gen_gl_q95', 'gen_gl_q99',
        'gen_copy_fraction_q95', 'threshold_q95', 'gen_nn_median', 'gen_nn_q95'
    ]
    cols = [c for c in preferred if c in df.columns]
    display(df.sort_values(['arch', 'dataset_size'])[cols] if cols else df.head())

## Training Loss Curves

This compares model depths at the same training-set size. DiT-L8 and DiT-L12 use their original 200k-update histories. For DiT-L16 at $N_{2D}=2^6,\ldots,2^{10}$, the plot uses the verified stage-4 continuation history through 300k updates; DiT-L16 falls back to 200k where no continuation exists. The curves are averaged over one 4,000-update learning-rate restart cycle for readability. Lower training loss can indicate easier fitting or memorization; it does not by itself establish sample quality or physical fidelity.


In [ ]:
def checkpoint_epoch(path: Path) -> int | None:
    m = re.search(r'checkpoint-epoch-(\d+)', str(path))
    return int(m.group(1)) if m else None


def metric_candidates(row: pd.Series) -> list[Path]:
    root = Path(str(row.get('checkpoint_dir', '') or ''))
    paths: list[Path] = []
    if root.exists():
        paths.extend(sorted(root.glob('metrics_epoch_*.json')))
        metrics_json = root / 'metrics.json'
        if metrics_json.exists():
            paths.append(metrics_json)
        for ckpt in sorted(root.glob('checkpoint-epoch-*')):
            paths.extend(sorted(ckpt.glob('metrics*.json')))
    return paths


def flatten_numeric(values: Any) -> np.ndarray:
    if values is None:
        return np.asarray([], dtype=float)
    out: list[float] = []

    def visit(x: Any) -> None:
        if x is None:
            return
        if isinstance(x, dict):
            for key in ('loss', 'value', 'mean', 'avg'):
                if key in x:
                    visit(x[key])
                    return
            return
        if isinstance(x, (list, tuple, np.ndarray)):
            for item in x:
                visit(item)
            return
        try:
            out.append(float(x))
        except (TypeError, ValueError):
            return

    visit(values)
    return np.asarray(out, dtype=float)


def read_latest_metrics(row: pd.Series) -> tuple[dict[str, Any], Path | None]:
    paths = metric_candidates(row)
    if not paths:
        return {}, None

    def score(path: Path) -> tuple[int, float]:
        epoch = checkpoint_epoch(path)
        return (epoch if epoch is not None else -1, path.stat().st_mtime)

    latest = max(paths, key=score)
    try:
        with latest.open() as f:
            return json.load(f), latest
    except Exception as exc:
        print('failed reading metrics:', latest, exc)
        return {}, latest


def downsample_xy(x: np.ndarray, y: np.ndarray, max_points: int = 1200) -> tuple[np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if len(x) <= max_points:
        return x, y
    idx = np.linspace(0, len(x) - 1, max_points, dtype=int)
    return x[idx], y[idx]


def cycle_average_epoch_loss(
    epoch_loss: np.ndarray,
    steps_per_epoch: int,
    restart_updates: int = 4000,
) -> tuple[np.ndarray, np.ndarray]:
    """Average roughly one LR-restart cycle without changing the update axis."""
    y = np.asarray(epoch_loss, dtype=float)
    if not len(y):
        return np.asarray([], dtype=float), np.asarray([], dtype=float)
    steps_per_epoch = max(1, int(steps_per_epoch))
    window = max(1, min(len(y), int(round(restart_updates / steps_per_epoch))))
    if window == 1:
        x = (np.arange(len(y), dtype=float) + 1.0) * steps_per_epoch
        return downsample_xy(x, y)
    kernel = np.ones(window, dtype=float) / window
    smooth = np.convolve(y, kernel, mode='valid')
    x = (np.arange(len(smooth), dtype=float) + 0.5 * (window + 1)) * steps_per_epoch
    return downsample_xy(x, smooth)


loss_by_run: dict[str, dict[str, Any]] = {}
loss_rows = []
if manifest_df.empty:
    display(Markdown('No manifest available; skipping loss audit.'))
else:
    for _, row in manifest_df.sort_values(['arch', 'dataset_size']).iterrows():
        metrics, metrics_path = read_latest_metrics(row)
        epoch_loss = flatten_numeric(metrics.get('epoch_loss'))
        batch_loss = flatten_numeric(metrics.get('loss', metrics.get('batch_loss')))
        epoch_lr = flatten_numeric(metrics.get('epoch_lr', metrics.get('lr')))
        run_name = str(row.get('run_name'))
        loss_by_run[run_name] = {
            'metrics': metrics,
            'metrics_path': metrics_path,
            'epoch_loss': epoch_loss,
            'batch_loss': batch_loss,
            'epoch_lr': epoch_lr,
        }
        steps_per_epoch = int(row.get('steps_per_epoch', 1) or 1)
        tail_window = max(1, min(len(epoch_loss), max(5, len(epoch_loss) // 20)))
        loss_rows.append({
            'arch': row.get('arch'),
            'arch_label': row.get('arch_label'),
            'run_name': run_name,
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': int(row.get('dataset_size')) if pd.notna(row.get('dataset_size')) else np.nan,
            'steps_per_epoch': steps_per_epoch,
            'gradient_accumulation_steps': int(row.get('gradient_accumulation_steps', 1) or 1),
            'metrics_path': rel(metrics_path) if metrics_path else None,
            'n_epoch_loss': len(epoch_loss),
            'final_epoch_loss': float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
            'best_epoch_loss': float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
            'tail_median_epoch_loss': float(np.nanmedian(epoch_loss[-tail_window:])) if len(epoch_loss) else np.nan,
            'epochs_completed': len(epoch_loss),
            'optimizer_updates_recorded': len(epoch_loss) * steps_per_epoch,
            'n_batch_loss': len(batch_loss),
            'final_batch_loss': float(batch_loss[-1]) if len(batch_loss) else np.nan,
        })

    loss_df = pd.DataFrame(loss_rows).sort_values(['arch', 'dataset_size'])

    # L16 continuation metrics through 300k live in isolated checkpoint directories.
    # Require the exact stage-4 checkpoint and a cumulative history close to 300k updates
    # before replacing the original 200k L16 curve.
    continue_loss_manifest_path = (
        PROJECT_DIR / 'local' / 'nf_generalize_fig2_dit_l16_continue' / 'manifest.json'
    )
    continue_loss_obj = read_json(continue_loss_manifest_path)
    continuation_loss_by_run: dict[str, dict[str, Any]] = {}
    continuation_loss_audit_rows = []
    if isinstance(continue_loss_obj, list):
        continue_loss_df = pd.DataFrame(continue_loss_obj)
        stage4 = continue_loss_df[
            (continue_loss_df.get('continue_stage', pd.Series(index=continue_loss_df.index, dtype=float)) == 4)
            & (continue_loss_df.get('target_total_updates', pd.Series(index=continue_loss_df.index, dtype=float)) == 300000)
        ].copy()
        for _, continue_row in stage4.sort_values('dataset_size').iterrows():
            run_name = str(continue_row.get('run_name'))
            expected_checkpoint = Path(str(continue_row.get('expected_checkpoint', '') or ''))
            metrics: dict[str, Any] = {}
            metrics_path: Path | None = None
            if expected_checkpoint.is_dir():
                checkpoint_row = continue_row.copy()
                checkpoint_row['checkpoint_dir'] = str(expected_checkpoint)
                metrics, metrics_path = read_latest_metrics(checkpoint_row)
                if not metrics:
                    metrics, metrics_path = read_latest_metrics(continue_row)
            epoch_loss = flatten_numeric(metrics.get('epoch_loss'))
            batch_loss = flatten_numeric(metrics.get('loss', metrics.get('batch_loss')))
            steps_per_epoch = max(1, int(continue_row.get('steps_per_epoch', 1) or 1))
            recorded_updates = len(epoch_loss) * steps_per_epoch
            use_300k = expected_checkpoint.is_dir() and recorded_updates >= 0.95 * 300000
            if use_300k:
                continuation_loss_by_run[run_name] = {
                    'metrics': metrics,
                    'metrics_path': metrics_path,
                    'epoch_loss': epoch_loss,
                    'batch_loss': batch_loss,
                    'epoch_lr': flatten_numeric(metrics.get('epoch_lr', metrics.get('lr'))),
                    'optimizer_updates_recorded': recorded_updates,
                    'source_label': '300k continuation',
                }
            continuation_loss_audit_rows.append({
                'run_name': run_name,
                'dataset_tag': continue_row.get('dataset_tag'),
                'expected_checkpoint_exists': expected_checkpoint.is_dir(),
                'optimizer_updates_recorded': recorded_updates,
                'using_300k_history': use_300k,
                'metrics_path': rel(metrics_path) if metrics_path else None,
            })
    continuation_loss_audit_df = pd.DataFrame(continuation_loss_audit_rows)
    if not continuation_loss_audit_df.empty:
        display(Markdown('### DiT-L16 300k loss-source audit'))
        display(continuation_loss_audit_df)
    loss_summary_cols = [
        'arch_label', 'dataset_tag', 'dataset_size', 'epochs_completed',
        'optimizer_updates_recorded', 'tail_median_epoch_loss', 'best_epoch_loss',
    ]
    display(loss_df[loss_summary_cols])

    plot_df = manifest_df[manifest_df['arch'].isin(LOSS_ARCHES) & manifest_df['dataset_tag'].isin(LOSS_TAGS)].copy()
    if plot_df.empty:
        plot_df = manifest_df.copy()

    if loss_df['n_epoch_loss'].sum() == 0 and loss_df['n_batch_loss'].sum() == 0:
        print('No training metrics JSON found yet.')
    else:
        selected_tags = [tag for tag in LOSS_TAGS if tag in set(plot_df['dataset_tag'])]
        fig, axes = plt.subplots(1, len(selected_tags), figsize=(18.0, 6.8), sharex=False, sharey=True)
        if len(selected_tags) == 1:
            axes = np.asarray([axes])
        for ax, dataset_tag in zip(axes, selected_tags):
            tag_df = plot_df[plot_df['dataset_tag'] == dataset_tag].copy()
            tag_df['arch_order'] = tag_df['arch'].map({name: i for i, name in enumerate(DIT_ARCH_ORDER)})
            panel_curves = []
            panel_sources = []
            for _, row in tag_df.sort_values('arch_order').iterrows():
                arch = str(row.get('arch'))
                run_name = str(row.get('run_name'))
                loss_source = loss_by_run.get(run_name, {})
                if arch == 'dit_l16' and run_name in continuation_loss_by_run:
                    loss_source = continuation_loss_by_run[run_name]
                epoch_loss = np.asarray(loss_source.get('epoch_loss', []), dtype=float)
                if not len(epoch_loss):
                    continue
                steps_per_epoch = max(1, int(row.get('steps_per_epoch', 1) or 1))
                x, y = cycle_average_epoch_loss(epoch_loss, steps_per_epoch)
                source_label = str(loss_source.get('source_label', '200k original'))
                panel_sources.append((arch, source_label, len(epoch_loss)))
                panel_curves.append((arch, x / 1000.0, y))
                ax.plot(
                    x / 1000.0, y, lw=2.8,
                    color=DIT_ARCH_COLORS.get(arch, '0.2'),
                    label=arch_label(arch),
                )
            dataset_size = dataset_size_from_tag(dataset_tag)
            dataset_exponent = int(round(np.log2(dataset_size)))
            l16_source = next((source for arch, source, _ in panel_sources if arch == 'dit_l16'), '200k original')
            representative_epochs = next((epochs for arch, _, epochs in panel_sources if arch == 'dit_l16'), 0)
            budget_note = 'L16: 300k' if l16_source == '300k continuation' else 'all: 200k'
            ax.set_title(
                f'$N_{{2D}}=2^{{{dataset_exponent}}}$\n{budget_note}; L16 {representative_epochs:,} epochs',
                fontsize=17, pad=12,
            )
            ax.set_xlabel('Optimizer updates (thousands)', fontsize=15)
            ax.set_yscale('log')
            ax.grid(alpha=0.18, which='major')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            if dataset_tag == 'd2p15' and panel_curves:
                inset = ax.inset_axes([0.46, 0.12, 0.50, 0.38])
                tail_values = []
                for arch, x_plot, y_plot in panel_curves:
                    inset.plot(x_plot, y_plot, lw=2.0, color=DIT_ARCH_COLORS.get(arch, '0.2'))
                    tail = y_plot[x_plot >= 20.0]
                    if len(tail):
                        tail_values.extend(tail.tolist())
                if tail_values:
                    y_min, y_max = float(np.nanmin(tail_values)), float(np.nanmax(tail_values))
                    padding = max(1e-6, 0.10 * (y_max - y_min))
                    inset.set_ylim(y_min - padding, y_max + padding)
                inset.set_xlim(20, max(float(np.nanmax(x)) for _, x, _ in panel_curves))
                inset.set_title('linear-scale zoom', fontsize=11, pad=3)
                inset.set_xlabel('updates (k)', fontsize=9)
                inset.tick_params(labelsize=8)
                inset.grid(alpha=0.16)
        axes[0].set_ylabel('Cycle-averaged training loss', fontsize=16)
        handles = [
            Line2D(
                [0], [0], color=DIT_ARCH_COLORS[a], lw=3.0,
                label=(arch_label(a) + (' (300k where available)' if a == 'dit_l16' else ' (200k)')),
            )
            for a in DIT_ARCH_ORDER
        ]
        fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.925), ncol=3, frameon=False, fontsize=14)
        fig.suptitle('DiT optimization with L16 continued to 300k', fontsize=21, fontweight='semibold', y=0.99)
        fig.text(
            0.5, 0.855,
            r'L16 uses 300k updates for $2^6$--$2^{10}$; L8, L12, and the $2^{15}$ panel use 200k.',
            ha='center', va='center', fontsize=14, color='0.30',
        )
        fig.subplots_adjust(left=0.07, right=0.99, bottom=0.12, top=0.70, wspace=0.10)
        out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_training_curves.png'
        QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)

        fig, ax = plt.subplots(figsize=(11.5, 6.4), constrained_layout=True)
        for arch in DIT_ARCH_ORDER:
            sub = loss_df[loss_df['arch'].astype(str) == arch].sort_values('dataset_size')
            if sub.empty:
                continue
            ax.plot(
                np.log2(sub['dataset_size'].astype(float)), sub['tail_median_epoch_loss'],
                marker=DIT_ARCH_MARKERS.get(arch, 'o'), ms=8.5, lw=2.8,
                color=DIT_ARCH_COLORS.get(arch, '0.2'), label=arch_label(arch),
            )
        exponents = sorted(np.log2(loss_df['dataset_size'].dropna().astype(float)).astype(int).unique())
        ax.set_xticks(exponents)
        ax.set_xticklabels([f'$2^{{{p}}}$' for p in exponents])
        ax.set_yscale('log')
        ax.set_xlabel(r'Training set size $N_{2D}$', fontsize=16)
        ax.set_ylabel('Median epoch loss over final 5%', fontsize=16)
        ax.set_title('Final training loss is driven mainly by dataset size', fontsize=19, pad=14)
        ax.grid(alpha=0.18, which='major')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(frameon=False, ncol=3, fontsize=13)
        summary_out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_tail_loss_by_data_size.png'
        fig.savefig(summary_out, bbox_inches='tight')
        plt.show()
        print('wrote', summary_out)

        selected_summary = loss_df[
            loss_df['arch'].isin(DIT_ARCH_ORDER) & loss_df['dataset_tag'].isin(selected_tags)
        ][['arch_label', 'dataset_tag', 'dataset_size', 'epochs_completed', 'tail_median_epoch_loss']]
        display(selected_summary.sort_values(['dataset_size', 'arch_label']))
        display(Markdown(
            '**Interpretation:** the first two panels intentionally show the extended 300k L16 history; '
            'the rightmost panel remains a fixed-200k comparison because no $2^{15}$ continuation exists. '
            'The inset is a linear-scale zoom of that rightmost panel after 20k updates. '
            'A low loss for a small dataset can reflect repeated exposure and memorization. '
            'The learning-rate traces are omitted because these runs share the same 4,000-update '
            'cosine-restart schedule; overlaying nine copies obscures the loss comparison.'
        ))

## Load Generated and Real Reference Slices

Each reference is loaded from the **training subset configured for that exact DiT run**, not from all available CAMELS maps. The config's source paths, `n_samples`, seed, and `zthin` define the subset. The notebook verifies that its exact slice count equals the manifest's $N_{2D}$.

Normalization uses the complete configured subset. For memory safety, general plotting may then take an even cap controlled by `DIT_MAX_REAL_REFERENCE_SLICES` (default 2,048; 0 means all). The nearest-training audit below refuses to run unless the complete configured subset is loaded.


In [ ]:
def npz_array_key(path: Path) -> str:
    with np.load(path) as data:
        return 'samples' if 'samples' in data.files else data.files[0]


def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        arr = np.asarray(data[key], dtype=np.float32)
    if SIMDIFF_EVAL_AVAILABLE:
        return as_nchw(arr)
    if arr.ndim == 3:
        return arr[:, None, :, :]
    if arr.ndim == 4 and arr.shape[1] in (1, 3):
        return arr
    if arr.ndim == 4 and arr.shape[-1] in (1, 3):
        return np.moveaxis(arr, -1, 1)
    raise ValueError(f'Could not interpret generated array shape {arr.shape}')


def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= int(limit):
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, int(limit), dtype=int)
    return arr[idx].copy()


def config_path_for(row: pd.Series) -> Path:
    raw = str(row.get('config', '') or row.get('config_path', '') or '')
    if raw:
        path = Path(raw)
        return path if path.is_absolute() else PROJECT_DIR / path
    return PROJECT_DIR / 'local' / SWEEP_NAME / 'configs' / f"{row['run_name']}.yaml"


def real_reference_cache_key(config_path: Path) -> str:
    config = yaml.safe_load(config_path.read_text()) or {}
    return json.dumps(config.get('data', config), sort_keys=True, default=str)


loaded: dict[str, dict[str, Any]] = {}
load_rows = []
real_reference_cache: dict[str, np.ndarray] = {}
real_reference_kind = (
    'model training subset; normalized from complete configured training set; '
    'evenly capped only after normalization'
)
if manifest_df.empty:
    display(Markdown('No manifest available; skipping generated/real loading.'))
elif not SIMDIFF_EVAL_AVAILABLE:
    display(Markdown(f'`simdiff_eval` unavailable, so real-reference diagnostics are skipped: `{SIMDIFF_EVAL_ERROR}`'))
else:
    for _, row in manifest_df.sort_values(['arch', 'dataset_size']).iterrows():
        sample_path = Path(row['sample_path_resolved'])
        if not sample_path.exists():
            continue
        cfg_path = config_path_for(row)
        try:
            reference_info = configured_training_reference_info(cfg_path)
            configured_slices = int(reference_info['configured_slices'])
            expected_slices = int(row.get('dataset_size'))
            reference_matches_manifest = configured_slices == expected_slices
            if not reference_matches_manifest:
                raise RuntimeError(
                    f"REAL REFERENCE MISMATCH for {row.get('run_name')}: config selects "
                    f"{configured_slices} slices but manifest says {expected_slices}"
                )
            generated = evenly_limit(load_npz_array(sample_path), MAX_GENERATED)
            cache_key = real_reference_cache_key(cfg_path)
            if cache_key not in real_reference_cache:
                real_reference_cache[cache_key] = load_real_reference_from_config(
                    cfg_path,
                    max_slices=MAX_REAL_REFERENCE_SLICES,
                )
            real = real_reference_cache[cache_key]
            run_name = str(row.get('run_name'))
            loaded[run_name] = {
                'spec': row,
                'real': real,
                'generated': generated,
                'sample_path': sample_path,
                'config_path': cfg_path,
                'reference_info': reference_info,
                'real_reference_kind': real_reference_kind,
            }
            load_rows.append({
                'arch': row.get('arch'),
                'arch_label': row.get('arch_label'),
                'run_name': run_name,
                'dataset_tag': row.get('dataset_tag'),
                'dataset_size': int(row.get('dataset_size')),
                'n_real_exact_model_subset': configured_slices,
                'n_real_sources': int(reference_info['n_sources']),
                'n_real_raw_simulations': int(reference_info['configured_raw_samples']),
                'real_zthin': int(reference_info['zthin']),
                'n_real_used_for_plot': len(real),
                'reference_complete_for_plot': len(real) == configured_slices,
                'reference_matches_manifest': reference_matches_manifest,
                'reference_selection': reference_info['selection'],
                'real_reference_kind': real_reference_kind,
                'n_generated_loaded': len(generated),
                'sample_path': rel(sample_path),
                'config_path': rel(cfg_path),
            })
        except Exception as exc:
            load_rows.append({
                'arch': row.get('arch'),
                'arch_label': row.get('arch_label'),
                'run_name': row.get('run_name'),
                'dataset_tag': row.get('dataset_tag'),
                'dataset_size': row.get('dataset_size'),
                'n_real_configured': row.get('dataset_size'),
                'n_real_loaded': 0,
                'real_reference_kind': real_reference_kind,
                'n_generated_loaded': 0,
                'error': repr(exc),
                'sample_path': rel(sample_path),
                'config_path': rel(cfg_path),
            })

loaded_df = pd.DataFrame(load_rows).sort_values(['arch', 'dataset_size']) if load_rows else pd.DataFrame()
display(loaded_df)
print('loaded DiT sample rows:', len(loaded))

## Generated Image Grids Across Data Size

A quick visual check across the DiT data-size sweep. These are not nearest-neighbor diagnostics; they just show what the generated fields look like as `N_2D` changes.


In [ ]:
def choose_bundles(tags: list[str], max_count: int | None = None, arch: str | None = None) -> list[dict[str, Any]]:
    tag_set = set(tags)
    bundles = [
        b for b in loaded.values()
        if str(b['spec'].get('dataset_tag')) in tag_set
        and (arch is None or str(b['spec'].get('arch')) == arch)
    ]
    bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
    if not bundles:
        bundles = [b for b in loaded.values() if arch is None or str(b['spec'].get('arch')) == arch]
        bundles = sorted(bundles, key=lambda b: int(b['spec']['dataset_size']))
    if max_count is not None and len(bundles) > max_count:
        keep = np.linspace(0, len(bundles) - 1, max_count, dtype=int)
        bundles = [bundles[i] for i in keep]
    return bundles


def plot_dit_image_grid(sample_index: int = 0, tags: list[str] = IMAGE_TAGS, arch: str = IMAGE_ARCH) -> Path | None:
    if not loaded:
        display(Markdown('No loaded DiT samples available for image grid.'))
        return None
    bundles = choose_bundles(tags, max_count=6, arch=arch)
    if not bundles:
        display(Markdown(f'No selected bundles available for `{arch_label(arch)}` image grid.'))
        return None

    values = []
    for b in bundles:
        values.append(b['generated'][min(sample_index, len(b['generated']) - 1), 0].ravel())
        values.append(b['real'][0, 0].ravel())
    flat = np.concatenate(values)
    vmin = float(np.nanquantile(flat, 0.005))
    vmax = float(np.nanquantile(flat, 0.995))

    fig, axes = plt.subplots(2, len(bundles), figsize=(2.7 * len(bundles), 5.8), squeeze=False, constrained_layout=True)
    for col, b in enumerate(bundles):
        row = b['spec']
        gen_idx = min(sample_index, len(b['generated']) - 1)
        axes[0, col].imshow(b['generated'][gen_idx, 0], cmap='viridis', vmin=vmin, vmax=vmax)
        axes[1, col].imshow(b['real'][0, 0], cmap='viridis', vmin=vmin, vmax=vmax)
        axes[0, col].set_title(dataset_size_label(int(row['dataset_size'])))
        for ax in axes[:, col]:
            ax.set_xticks([])
            ax.set_yticks([])
    axes[0, 0].set_ylabel('generated', fontsize=15, fontweight='bold')
    axes[1, 0].set_ylabel('real reference', fontsize=15, fontweight='bold')
    fig.suptitle(f'{arch_label(arch)} generated maps across training-set size', y=1.03)
    out = QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_generated_image_grid.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

image_grid_path = plot_dit_image_grid(sample_index=int(os.environ.get('DIT_IMAGE_SAMPLE_INDEX', '0')))

## Generated Samples Versus Nearest Training Slices

This is the requested visual copy check for the DiT runs. For each selected generated map, the notebook searches the **complete configured training subset for that exact model** in pixel mean-squared error, then shows the generated map, its closest training slice, and their absolute difference.

The summary covers DiT-L8, DiT-L12, and DiT-L16 at $N_{2D}=2^6,\ldots,2^{10}$. A second panel shows four DiT-L16 generated maps at $N_{2D}=2^8$ so the visibly noisy sample can be checked as a possible fluke. This pixel-space plot is a visual audit; PCA and SSCD remain the primary full-sample novelty metrics.


In [ ]:
DIT_NN_ARCHES = parse_csv_env(
    'DIT_NN_ARCHES', 'dit_l8,dit_base,dit_l16'
)
DIT_NN_TAGS = parse_csv_env('DIT_NN_TAGS', 'd2p06,d2p07,d2p08,d2p09,d2p10')
DIT_NN_MAX_GENERATED = int(os.environ.get('DIT_NN_MAX_GENERATED', '4'))
DIT_NN_SUMMARY_GENERATED_INDEX = int(os.environ.get('DIT_NN_SUMMARY_GENERATED_INDEX', '0'))
DIT_NN_AUDIT_ARCH = os.environ.get('DIT_NN_AUDIT_ARCH', 'dit_l16')
DIT_NN_AUDIT_TAG = os.environ.get('DIT_NN_AUDIT_TAG', 'd2p08')


def require_complete_training_reference(bundle: dict[str, Any]) -> int:
    configured_slices = int(bundle['reference_info']['configured_slices'])
    loaded_slices = len(bundle['real'])
    if loaded_slices != configured_slices:
        raise RuntimeError(
            'nearest-training search requires the complete configured subset: '
            f"run={bundle['spec']['run_name']} loaded={loaded_slices} "
            f'configured={configured_slices}. Increase DIT_MAX_REAL_REFERENCE_SLICES '
            'or restrict DIT_NN_TAGS to sizes at or below the cap.'
        )
    return configured_slices


def build_dit_nearest_training_audit(
    arches: list[str] = DIT_NN_ARCHES,
    tags: list[str] = DIT_NN_TAGS,
) -> tuple[pd.DataFrame, dict[str, dict[str, Any]]]:
    records = []
    by_run = {}
    for arch in arches:
        for bundle in choose_bundles(tags, arch=arch):
            row = bundle['spec']
            configured_slices = require_complete_training_reference(bundle)
            matches = nearest_training_matches(
                bundle['generated'],
                bundle['real'],
                max_generated=DIT_NN_MAX_GENERATED,
                max_training=None,
                training_chunk=256,
            )
            by_run[str(row['run_name'])] = {'bundle': bundle, 'matches': matches}
            for i, generated_index in enumerate(matches['generated_index']):
                records.append({
                    'arch': str(row['arch']),
                    'run_name': str(row['run_name']),
                    'dataset_tag': str(row['dataset_tag']),
                    'dataset_size': int(row['dataset_size']),
                    'generated_index': int(generated_index),
                    'nearest_training_index': int(matches['nearest_training_index'][i]),
                    'nearest_mse': float(matches['nearest_mse'][i]),
                    'nearest_rmse': float(matches['nearest_rmse'][i]),
                    'nearest_cosine': float(matches['nearest_cosine'][i]),
                    'n_training_searched': configured_slices,
                    'reference_is_complete': True,
                    'sample_path': rel(bundle['sample_path']),
                    'config_path': rel(bundle['config_path']),
                })
    audit = pd.DataFrame(records)
    if not audit.empty:
        audit = audit.sort_values(['arch', 'dataset_size', 'generated_index'])
        TABLE_DIR.mkdir(parents=True, exist_ok=True)
        out = TABLE_DIR / 'nf_generalize_fig2_dit_pixel_nearest_training.csv'
        audit.to_csv(out, index=False)
        print('wrote', out)
        display(audit)
    return audit, by_run


def nearest_plot_limits(items: list[tuple[np.ndarray, np.ndarray]]) -> tuple[float, float, float]:
    fields = np.concatenate([image.ravel() for pair in items for image in pair])
    differences = np.concatenate([np.abs(first - second).ravel() for first, second in items])
    return (
        float(np.nanquantile(fields, 0.005)),
        float(np.nanquantile(fields, 0.995)),
        float(np.nanquantile(differences, 0.995)),
    )


def plot_dit_nearest_training_summary(
    audit: pd.DataFrame,
    by_run: dict[str, dict[str, Any]],
    arch: str,
) -> Path | None:
    selected_arch = audit[audit['arch'] == arch]
    if selected_arch.empty:
        display(Markdown(f'No `{arch}` samples are available for the nearest-training summary.'))
        return None
    selected = selected_arch[
        selected_arch['generated_index'] == DIT_NN_SUMMARY_GENERATED_INDEX
    ].copy()
    if selected.empty:
        selected = selected_arch.groupby('run_name', as_index=False).head(1)
    selected = selected.sort_values('dataset_size')
    items = []
    for _, record in selected.iterrows():
        bundle = by_run[str(record['run_name'])]['bundle']
        generated = bundle['generated'][int(record['generated_index']), 0]
        training = bundle['real'][int(record['nearest_training_index']), 0]
        items.append((record, generated, training))
    vmin, vmax, diff_vmax = nearest_plot_limits([(gen, train) for _, gen, train in items])

    fig, axes = plt.subplots(
        3, len(items), figsize=(3.05 * len(items), 8.9),
        squeeze=False, constrained_layout=True,
    )
    for col, (record, generated, training) in enumerate(items):
        axes[0, col].imshow(generated, cmap='viridis', vmin=vmin, vmax=vmax)
        axes[1, col].imshow(training, cmap='viridis', vmin=vmin, vmax=vmax)
        axes[2, col].imshow(np.abs(generated - training), cmap='magma', vmin=0, vmax=diff_vmax)
        axes[0, col].set_title(dataset_size_label(int(record['dataset_size'])), fontweight='bold')
        axes[2, col].text(
            0.5, 0.02,
            f"MSE={record['nearest_mse']:.3g}; cos={record['nearest_cosine']:.3f}",
            transform=axes[2, col].transAxes, ha='center', va='bottom', fontsize=9,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=2),
        )
        for ax in axes[:, col]:
            ax.set_xticks([])
            ax.set_yticks([])
    for row_index, label in enumerate(('generated', 'nearest training', 'absolute difference')):
        axes[row_index, 0].set_ylabel(label, fontsize=14, fontweight='bold')
    fig.suptitle(
        f'{arch_label(arch)} generated samples versus nearest training slices',
        fontsize=21, fontweight='bold',
    )
    out = QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_generated_vs_nearest_training.png'
    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


def plot_dit_nearest_training_fluke_audit(
    audit: pd.DataFrame,
    by_run: dict[str, dict[str, Any]],
    arch: str = DIT_NN_AUDIT_ARCH,
    audit_tag: str = DIT_NN_AUDIT_TAG,
) -> Path | None:
    selected = audit[
        (audit['arch'] == arch) & (audit['dataset_tag'] == audit_tag)
    ].sort_values('generated_index')
    if selected.empty:
        display(Markdown(f'No DiT nearest-training records are available for `{audit_tag}`.'))
        return None
    run_name = str(selected.iloc[0]['run_name'])
    bundle = by_run[run_name]['bundle']
    items = []
    for _, record in selected.iterrows():
        generated = bundle['generated'][int(record['generated_index']), 0]
        training = bundle['real'][int(record['nearest_training_index']), 0]
        items.append((record, generated, training))
    vmin, vmax, diff_vmax = nearest_plot_limits([(gen, train) for _, gen, train in items])

    fig, axes = plt.subplots(
        len(items), 3, figsize=(9.6, 3.0 * len(items)),
        squeeze=False, constrained_layout=True,
    )
    for row_index, (record, generated, training) in enumerate(items):
        axes[row_index, 0].imshow(generated, cmap='viridis', vmin=vmin, vmax=vmax)
        axes[row_index, 1].imshow(training, cmap='viridis', vmin=vmin, vmax=vmax)
        axes[row_index, 2].imshow(np.abs(generated - training), cmap='magma', vmin=0, vmax=diff_vmax)
        axes[row_index, 0].set_ylabel(
            f"generated {int(record['generated_index'])}", fontsize=12, fontweight='bold'
        )
        axes[row_index, 2].text(
            0.5, 0.02,
            f"MSE={record['nearest_mse']:.3g}; cos={record['nearest_cosine']:.3f}",
            transform=axes[row_index, 2].transAxes, ha='center', va='bottom', fontsize=9,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=2),
        )
        for ax in axes[row_index]:
            ax.set_xticks([])
            ax.set_yticks([])
    for col, title in enumerate(('generated', 'nearest training', 'absolute difference')):
        axes[0, col].set_title(title, fontsize=14, fontweight='bold')
    n_value = int(selected.iloc[0]['dataset_size'])
    fig.suptitle(
        f'{arch_label(arch)} {dataset_size_label(n_value)}: '
        f'{len(items)} generated-sample nearest-training audit',
        fontsize=19, fontweight='bold',
    )
    out = QUICKCHECK_DIR / (
        f'nf_generalize_fig2_{arch}_{audit_tag}_nearest_training_audit.png'
    )
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


dit_pixel_nn_df, dit_pixel_nn_by_run = build_dit_nearest_training_audit()
dit_pixel_nn_summary_paths = {
    arch: plot_dit_nearest_training_summary(dit_pixel_nn_df, dit_pixel_nn_by_run, arch)
    for arch in DIT_NN_ARCHES
}
dit_pixel_nn_fluke_path = plot_dit_nearest_training_fluke_audit(dit_pixel_nn_df, dit_pixel_nn_by_run)


## SSCD Distribution Distance: Are Novel Samples In Distribution?

Nearest-neighbor novelty can assign a high score to a visibly bad sample simply because it is far from every training image. This section adds the requested distribution-level check.

For each run, it loads the SSCD embeddings already cached by the full nearest-neighbor analysis, compares generated embeddings with **heldout real** embeddings, and reports a Fréchet feature distance. Because finite samples have a nonzero Fréchet distance even when both sets are real, the generated-to-heldout value is normalized by a same-size **real-vs-real split baseline**.

Read the ratio as follows:

- near 1: generated-to-real separation is comparable to finite real-split variability;
- substantially above 1: generated samples are farther from the real distribution, so a high novelty score may be invalid;
- below 1: not automatically better, because copying or low diversity can also reduce a distribution distance.

This is an in-distribution diagnostic, not a replacement for nearest-training plots, the one-point PDF, or $P(k)$.


In [ ]:
SSCD_CACHE_DIR = RESULTS_DIR / 'cache' / 'sscd_full_nn'
DIT_DISTRIBUTION_ARCHES = parse_csv_env(
    'DIT_DISTRIBUTION_ARCHES', 'dit_l8,dit_base,dit_l16'
)
DIT_DISTRIBUTION_TAGS = parse_csv_env(
    'DIT_DISTRIBUTION_TAGS',
    'd2p06,d2p07,d2p08,d2p09,d2p10,d2p11,d2p12,d2p13,d2p14,d2p15',
)
DIT_DISTRIBUTION_COMPONENTS = int(os.environ.get('DIT_DISTRIBUTION_COMPONENTS', '64'))
DIT_DISTRIBUTION_SEED = int(os.environ.get('DIT_DISTRIBUTION_SEED', str(SEED)))


def find_sscd_embedding_cache(run_name: str, kind: str) -> Path | None:
    pattern = f'{run_name}_{kind}_{SAMPLE_LABEL}_seed{SEED}_*.pt'
    matches = sorted(
        SSCD_CACHE_DIR.glob(pattern),
        key=lambda path: (path.stat().st_mtime, path.name),
    )
    return matches[-1] if matches else None


def load_sscd_embedding_cache(path: Path) -> np.ndarray:
    import torch

    try:
        payload = torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        payload = torch.load(path, map_location='cpu')
    tensor = payload['embeddings'] if isinstance(payload, dict) else payload
    features = np.asarray(tensor.detach().cpu(), dtype=np.float64)
    if features.ndim != 2 or len(features) < 4:
        raise ValueError(f'invalid SSCD embedding cache {path}: shape={features.shape}')
    return features


def project_to_real_pca(
    real_features: np.ndarray,
    generated_features: np.ndarray,
    max_components: int,
) -> tuple[np.ndarray, np.ndarray, int]:
    center = real_features.mean(axis=0, keepdims=True)
    centered_real = real_features - center
    centered_generated = generated_features - center
    max_rank = min(
        int(max_components),
        centered_real.shape[0] - 2,
        centered_real.shape[1],
    )
    if max_rank < 1:
        raise ValueError('not enough heldout real embeddings for PCA projection')
    _, _, right_vectors = np.linalg.svd(centered_real, full_matrices=False)
    basis = right_vectors[:max_rank].T
    return centered_real @ basis, centered_generated @ basis, max_rank


def evaluate_sscd_distribution_distance() -> pd.DataFrame:
    rows = []
    if manifest_df.empty:
        display(Markdown('No manifest is available for the SSCD distribution-distance audit.'))
        return pd.DataFrame()

    selected = manifest_df[
        manifest_df['arch'].isin(DIT_DISTRIBUTION_ARCHES)
        & manifest_df['dataset_tag'].isin(DIT_DISTRIBUTION_TAGS)
    ].sort_values(['arch', 'dataset_size'])
    for _, run in selected.iterrows():
        run_name = str(run['run_name'])
        heldout_path = find_sscd_embedding_cache(run_name, 'heldout')
        generated_path = find_sscd_embedding_cache(run_name, 'generated')
        if heldout_path is None or generated_path is None:
            rows.append({
                'arch': str(run['arch']),
                'arch_label': arch_label(run['arch']),
                'run_name': run_name,
                'dataset_tag': str(run['dataset_tag']),
                'dataset_size': int(run['dataset_size']),
                'status': 'missing SSCD cache',
                'heldout_cache': rel(heldout_path),
                'generated_cache': rel(generated_path),
            })
            continue

        heldout = load_sscd_embedding_cache(heldout_path)
        generated = load_sscd_embedding_cache(generated_path)
        heldout_projected, generated_projected, rank = project_to_real_pca(
            heldout,
            generated,
            DIT_DISTRIBUTION_COMPONENTS,
        )
        n_eval = min(len(generated_projected), len(heldout_projected) // 2)
        if n_eval < 2:
            raise ValueError(f'not enough equal-size samples for {run_name}')

        rng = np.random.default_rng(DIT_DISTRIBUTION_SEED)
        real_indices = rng.permutation(len(heldout_projected))[: 2 * n_eval]
        generated_indices = rng.permutation(len(generated_projected))[:n_eval]
        real_first = heldout_projected[real_indices[:n_eval]]
        real_second = heldout_projected[real_indices[n_eval:]]
        generated_eval = generated_projected[generated_indices]

        real_baseline = real_split_frechet_baseline(
            np.concatenate([real_first, real_second], axis=0),
            seed=DIT_DISTRIBUTION_SEED,
        )
        generated_distance = frechet_feature_distance(generated_eval, real_first)
        baseline_distance = float(real_baseline['distance'])
        ratio = generated_distance / max(baseline_distance, 1e-12)
        rows.append({
            'arch': str(run['arch']),
            'arch_label': arch_label(run['arch']),
            'run_name': run_name,
            'dataset_tag': str(run['dataset_tag']),
            'dataset_size': int(run['dataset_size']),
            'status': 'ok',
            'feature_space': 'SSCD projected on heldout-real PCA',
            'pca_rank': int(rank),
            'n_generated_eval': int(n_eval),
            'n_heldout_eval': int(n_eval),
            'generated_to_heldout_frechet': float(generated_distance),
            'real_split_frechet': baseline_distance,
            'generated_to_heldout_over_real_split': float(ratio),
            'heldout_cache': rel(heldout_path),
            'generated_cache': rel(generated_path),
        })

    frame = pd.DataFrame(rows)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    out = TABLE_DIR / 'nf_generalize_fig2_dit_sscd_frechet_distribution_distance.csv'
    frame.to_csv(out, index=False)
    print('wrote', out)
    display(frame)
    return frame


def plot_sscd_distribution_distance(frame: pd.DataFrame) -> Path | None:
    required_columns = {
        'status',
        'generated_to_heldout_over_real_split',
        'dataset_size',
        'arch',
    }
    if frame.empty or not required_columns.issubset(frame.columns):
        display(Markdown(
            '**SSCD distribution-distance plot skipped:** cached heldout and generated '
            'SSCD embeddings were not found. Run the SSCD full-nearest-neighbor analysis first.'
        ))
        return None
    valid = frame[
        (frame['status'] == 'ok')
        & pd.to_numeric(
            frame['generated_to_heldout_over_real_split'], errors='coerce'
        ).notna()
    ].copy()
    if valid.empty:
        display(Markdown(
            '**SSCD distribution-distance plot skipped:** cached heldout and generated '
            'SSCD embeddings were not found. Run the SSCD full-nearest-neighbor analysis first.'
        ))
        return None

    fig, ax = plt.subplots(figsize=(11.8, 6.8), constrained_layout=True)
    for arch in DIT_ARCH_ORDER:
        sub = valid[valid['arch'] == arch].sort_values('dataset_size')
        if sub.empty:
            continue
        ax.plot(
            np.log2(sub['dataset_size'].astype(float)),
            sub['generated_to_heldout_over_real_split'].astype(float),
            color=DIT_ARCH_COLORS[arch],
            marker=DIT_ARCH_MARKERS[arch],
            lw=2.8,
            ms=8,
            label=arch_label(arch),
        )
    exponents = sorted(np.log2(valid['dataset_size'].astype(float)).astype(int).unique())
    ax.set_xticks(exponents)
    ax.set_xticklabels([f'$2^{{{exponent}}}$' for exponent in exponents])
    ax.axhline(1.0, color='0.25', lw=1.8, ls='--', label='real-vs-real baseline')
    ax.set_yscale('log')
    ax.set_xlabel(r'Training images $N_{2D}$')
    ax.set_ylabel('Generated-to-heldout / real-split Fréchet distance')
    ax.set_title('SSCD distribution distance: novelty does not imply validity', pad=14)
    ax.grid(alpha=0.18, which='both')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False, ncol=2)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_frechet_distribution_distance.png'
    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


dit_sscd_distribution_df = evaluate_sscd_distribution_distance()
dit_sscd_distribution_plot = plot_sscd_distribution_distance(dit_sscd_distribution_df)


## One-point and P(k) Agreement Across Data Size

These panels ask a different question from memorization: do generated samples match real CAMELS summary statistics? The black curve is the reference from **that model's configured training subset**, not all CAMELS maps. The panel subtitle reports how many configured slices are used after the memory cap. Small-data runs can look good here by copying training slices, so read this together with the PCA/SSCD novelty curves and nearest-training plots.


In [ ]:
def plot_dit_onepoint_pk(tags: list[str] = DETAIL_TAGS, arch: str = DETAIL_ARCH) -> Path | None:
    if not loaded:
        display(Markdown('No loaded DiT samples available for one-point/P(k) plots.'))
        return None
    if not SIMDIFF_EVAL_AVAILABLE:
        display(Markdown('`simdiff_eval` unavailable; cannot compute one-point/P(k) diagnostics.'))
        return None

    bundles = choose_bundles(tags, max_count=5, arch=arch)
    if not bundles:
        display(Markdown(f'No selected `{arch_label(arch)}` bundles available for one-point/P(k) plots.'))
        return None

    n = len(bundles)
    fig, axes = plt.subplots(2, n, figsize=(max(4.2 * n, 8.6), 7.4), squeeze=False, constrained_layout=True)
    rows = []
    for col, bundle in enumerate(bundles):
        row = bundle['spec']
        real = bundle['real']
        generated = bundle['generated']

        rh = field_histogram(real, bins=140)
        gh = field_histogram(generated, bins=140)
        edges = np.asarray(rh['bin_edges'])
        centers = 0.5 * (edges[:-1] + edges[1:])
        axes[0, col].plot(centers, rh['hist'], color='black', lw=2.2, label='model training subset')
        axes[0, col].plot(centers, gh['hist'], color=DIT_ARCH_COLORS.get(arch, '#0072B2'), lw=2.0, label=f'{arch_label(arch)} generated')
        axes[0, col].set_yscale('log')
        configured_slices = int(bundle['reference_info']['configured_slices'])
        axes[0, col].set_title(
            f"{dataset_size_label(configured_slices)} one-point\n"
            f"black: {len(real):,}/{configured_slices:,} training slices"
        )
        axes[0, col].set_xlabel('Normalized field value')
        if col == 0:
            axes[0, col].set_ylabel('Pixel PDF')
            axes[0, col].legend(frameon=False, loc='upper right')

        pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
        pk_gen, _ = batch_power_spectra(generated, nbins=PK_NBINS)
        mean_real = np.clip(np.nanmean(pk_real, axis=0), 1e-30, None)
        mean_gen = np.nanmean(pk_gen, axis=0)
        ratio = mean_gen / mean_real
        axes[1, col].plot(kbins, ratio, marker='o', ms=4.4, lw=2.0, color=DIT_ARCH_COLORS.get(arch, '#0072B2'))
        axes[1, col].axhline(1.0, color='0.25', ls='--', lw=1.4)
        upper = max(2.0, float(np.nanquantile(ratio, 0.98)) * 1.15 if np.isfinite(ratio).any() else 2.0)
        axes[1, col].set_ylim(0, upper)
        axes[1, col].set_title('Mean $P(k)$ ratio')
        axes[1, col].set_xlabel('$k$ bin')
        if col == 0:
            axes[1, col].set_ylabel('generated / real')

        finite = np.isfinite(ratio)
        sample_path = bundle.get('sample_path')
        config_path = bundle.get('config_path')
        rows.append({
            'arch': row.get('arch'),
            'arch_label': row.get('arch_label'),
            'run_name': row.get('run_name'),
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': int(row['dataset_size']),
            'sample_path': rel(sample_path),
            'config_path': rel(config_path),
            'generated_shape': tuple(generated.shape),
            'real_shape': tuple(real.shape),
            'generated_mean': float(np.nanmean(generated)),
            'real_mean': float(np.nanmean(real)),
            'generated_std': float(np.nanstd(generated)),
            'real_std': float(np.nanstd(real)),
            'generated_min': float(np.nanmin(generated)),
            'real_min': float(np.nanmin(real)),
            'generated_max': float(np.nanmax(generated)),
            'real_max': float(np.nanmax(real)),
            'pk_ratio_median': float(np.nanmedian(ratio[finite])) if finite.any() else np.nan,
            'pk_ratio_min': float(np.nanmin(ratio[finite])) if finite.any() else np.nan,
            'pk_ratio_max': float(np.nanmax(ratio[finite])) if finite.any() else np.nan,
            'max_abs_pk_ratio_minus_1': float(np.nanmax(np.abs(ratio[finite] - 1.0))) if finite.any() else np.nan,
        })

        for ax in axes[:, col]:
            ax.grid(alpha=0.18)
            for spine in ['top', 'right']:
                ax.spines[spine].set_visible(False)

    fig.suptitle(f'{arch_label(arch)} physical-statistics check', y=1.03)
    out = QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_detail_onepoint_pk.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)

    fidelity_summary = pd.DataFrame(rows).sort_values(['arch', 'dataset_size'])
    display(fidelity_summary)
    table_out = TABLE_DIR / f'nf_generalize_fig2_{arch}_fidelity_summary.csv'
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    fidelity_summary.to_csv(table_out, index=False)
    print('wrote', table_out)
    return out

fidelity_plot_path = plot_dit_onepoint_pk()

## Focused DiT Small-Data Sanity Checks

The DiT curves show suspicious small-data behavior, especially for the deeper models. This section compares the same $N_{2D}=2^6,2^7,2^8,2^9,2^{10}$ runs across DiT-L8, DiT-L12/base, and DiT-L16.

The goal is to separate three possibilities:
1. **real architecture behavior:** deeper DiTs genuinely leave the training-neighbor regime earlier;
2. **metric-specific artifact:** PCA and SSCD disagree, so the effect depends on the embedding;
3. **bad-generation artifact:** loss, images, one-point PDFs, or $P(k)$ look poor even when the generalization score is high.

In [ ]:
SANITY_ARCHES = parse_csv_env('DIT_SANITY_ARCHES', 'dit_l8,dit_base,dit_l16')
SANITY_TAGS = parse_csv_env('DIT_SANITY_TAGS', 'd2p06,d2p07,d2p08,d2p09,d2p10')
SANITY_SAMPLE_INDEX = int(os.environ.get('DIT_SANITY_SAMPLE_INDEX', '0'))


def small_data_metric_comparison(
    arches: list[str] = SANITY_ARCHES,
    tags: list[str] = SANITY_TAGS,
) -> pd.DataFrame:
    frames = []
    for feature, metrics in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
        if metrics.empty:
            continue
        tmp = ensure_arch_columns(metrics)
        if 'dataset_tag' not in tmp.columns:
            continue
        sub = tmp[tmp['arch'].astype(str).isin(arches) & tmp['dataset_tag'].astype(str).isin(tags)].copy()
        if sub.empty:
            continue
        keep = [
            'run_name', 'arch', 'arch_label', 'dataset_tag', 'dataset_size',
            'n_generated', 'n_train',
            'gen_gl_q90', 'gen_gl_q95', 'gen_gl_q99',
            'gen_copy_fraction_q90', 'gen_copy_fraction_q95', 'gen_copy_fraction_q99',
            'threshold_q90', 'threshold_q95', 'threshold_q99',
            'gen_nn_median', 'gen_nn_q95',
        ]
        sub = sub[[c for c in keep if c in sub.columns]]
        sub.insert(0, 'feature', feature)
        frames.append(sub)

    if not frames:
        display(Markdown('No PCA/SSCD metrics found for the small-data sanity check.'))
        return pd.DataFrame()

    comp = pd.concat(frames, ignore_index=True)
    comp['arch'] = comp['arch'].astype(str).replace({'dit': 'dit_base'})
    comp['arch_order'] = comp['arch'].map({a: i for i, a in enumerate(DIT_ARCH_ORDER)}).fillna(999)
    comp = comp.sort_values(['arch_order', 'dataset_size', 'feature']).drop(columns=['arch_order'])
    display(comp)

    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    out = TABLE_DIR / 'nf_generalize_fig2_dit_small_data_pca_sscd_comparison.csv'
    comp.to_csv(out, index=False)
    print('wrote', out)
    return comp


small_data_metric_comparison_df = small_data_metric_comparison()

if not small_data_metric_comparison_df.empty and 'gen_gl_q95' in small_data_metric_comparison_df.columns:
    valid_arches = [a for a in SANITY_ARCHES if a in set(small_data_metric_comparison_df['arch'].astype(str))]
    fig, axes = plt.subplots(1, len(valid_arches), figsize=(5.0 * max(1, len(valid_arches)), 4.8), sharey=True, constrained_layout=True)
    if len(valid_arches) == 1:
        axes = [axes]
    for ax, arch in zip(axes, valid_arches):
        arch_df = small_data_metric_comparison_df[small_data_metric_comparison_df['arch'].astype(str) == arch].copy()
        for feature, color, marker in [('PCA', '#0072B2', 'o'), ('SSCD', '#D55E00', 's')]:
            sub = arch_df[arch_df['feature'] == feature].sort_values('dataset_size')
            if sub.empty:
                continue
            ax.plot(
                sub['dataset_size'], sub['gen_gl_q95'],
                marker=marker, ms=7.8, lw=2.5, color=color,
                label=f'{feature} q95',
            )
        ticks = sorted(arch_df['dataset_size'].dropna().astype(int).unique())
        ax.axhline(0.5, color='0.35', lw=1.4, ls=':')
        ax.set_xscale('log', base=2)
        ax.set_xticks(ticks)
        ax.set_xticklabels([dataset_size_label(t) for t in ticks])
        ax.set_ylim(-0.04, 1.04)
        ax.set_title(arch_label(arch))
        ax.set_xlabel(r'Training set size $N_{2D}$')
        ax.grid(alpha=0.22)
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
    axes[0].set_ylabel('Generalization score')
    handles, labels = axes[-1].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.06), ncol=2, frameon=False)
    fig.suptitle('Small-data sanity check: PCA vs SSCD generalization', y=1.16)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_small_data_pca_sscd_q95_by_depth.png'
    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
else:
    display(Markdown('No `gen_gl_q95` column available for focused PCA-vs-SSCD plot.'))

In [ ]:
small_data_image_paths = {}
for arch in SANITY_ARCHES:
    display(Markdown(f'### {arch_label(arch)} generated maps for $2^6$--$2^{{10}}$'))
    small_data_image_paths[arch] = plot_dit_image_grid(
        sample_index=SANITY_SAMPLE_INDEX,
        tags=SANITY_TAGS,
        arch=arch,
    )

In [ ]:
small_data_fidelity_paths = {}
for arch in SANITY_ARCHES:
    display(Markdown(f'### {arch_label(arch)} one-point PDF and $P(k)$ for $2^6$--$2^{{10}}$'))
    small_data_fidelity_paths[arch] = plot_dit_onepoint_pk(tags=SANITY_TAGS, arch=arch)

In [ ]:
def plot_small_data_loss_curves(
    arch: str,
    tags: list[str] = SANITY_TAGS,
) -> Path | None:
    if manifest_df.empty or 'loss_by_run' not in globals():
        display(Markdown('No manifest/loss cache available for focused loss curves.'))
        return None

    plot_df = manifest_df[
        (manifest_df['arch'].astype(str) == arch)
        & manifest_df['dataset_tag'].astype(str).isin(tags)
    ].copy().sort_values('dataset_size')
    if plot_df.empty:
        display(Markdown(f'No manifest rows found for `{arch_label(arch)}` focused loss curves.'))
        return None

    if 'loss_df' in globals() and isinstance(loss_df, pd.DataFrame) and not loss_df.empty:
        focus_loss = loss_df[
            (loss_df['arch'].astype(str) == arch)
            & loss_df['dataset_tag'].astype(str).isin(tags)
        ].copy()
        focus_cols = [c for c in [
            'arch_label', 'dataset_tag', 'dataset_size', 'n_epoch_loss',
            'final_epoch_loss', 'best_epoch_loss', 'n_batch_loss', 'final_batch_loss',
            'gradient_accumulation_steps', 'metrics_path',
        ] if c in focus_loss.columns]
        if focus_cols:
            display(focus_loss.sort_values('dataset_size')[focus_cols])

    if not any(
        len(np.asarray(loss_by_run.get(str(row['run_name']), {}).get('epoch_loss', [])))
        or len(np.asarray(loss_by_run.get(str(row['run_name']), {}).get('batch_loss', [])))
        for _, row in plot_df.iterrows()
    ):
        display(Markdown(f'No loss arrays found for `{arch_label(arch)}` focused loss curves.'))
        return None

    palette = ['#3B0F70', '#365C8D', '#1F968B', '#73D055', '#FDE725']
    with plt.rc_context({
        'font.family': 'serif',
        'font.serif': ['DejaVu Serif'],
        'mathtext.fontset': 'dejavuserif',
        'font.size': 15,
        'axes.labelsize': 18,
        'axes.titlesize': 20,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'legend.fontsize': 14,
    }):
        fig, axes = plt.subplots(1, 2, figsize=(14.8, 5.8))
        fig.subplots_adjust(left=0.075, right=0.985, bottom=0.20, top=0.67, wspace=0.16)
        for color, (_, row) in zip(palette, plot_df.iterrows()):
            run_name = str(row.get('run_name'))
            info = loss_by_run.get(run_name, {})
            label = dataset_size_label(int(row['dataset_size']))
            steps_per_epoch = max(1, int(row.get('steps_per_epoch', 1) or 1))
            grad_accum = max(1, int(row.get('gradient_accumulation_steps', 1) or 1))

            epoch_loss = np.asarray(info.get('epoch_loss', []), dtype=float)
            if len(epoch_loss):
                optimizer_updates = np.arange(len(epoch_loss), dtype=float) * steps_per_epoch
                x, y = downsample_xy(optimizer_updates, epoch_loss, max_points=900)
                axes[0].plot(x, y, lw=2.2, color=color, label=label)

            batch_loss = np.asarray(info.get('batch_loss', []), dtype=float)
            if len(batch_loss):
                micro_updates = np.arange(len(batch_loss), dtype=float)
                # Convert recorded micro-batches to optimizer updates: micro_updates / grad_accum.
                x = micro_updates / grad_accum
                y = batch_loss.copy()
                window = max(1, len(y) // 900)
                if window > 1:
                    kernel = np.ones(window, dtype=float) / window
                    y = np.convolve(y, kernel, mode='valid')
                    x = x[:len(y)] + 0.5 * (window - 1) / grad_accum
                x, y = downsample_xy(x, y, max_points=900)
                axes[1].plot(x, y, lw=2.0, color=color, label=label)

        axes[0].set_title('Epoch-mean denoising loss', pad=10)
        axes[0].set_ylabel('Training loss')
        axes[1].set_title('Smoothed batch denoising loss', pad=10)
        for ax in axes:
            ax.set_xlabel('Optimizer update')
            ax.set_yscale('log')
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(width=1.1, length=5)

        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(
                handles, labels, title=r'Training images $N_{2D}$',
                loc='upper center', bbox_to_anchor=(0.5, 0.86),
                ncol=len(labels), frameon=False,
            )
        fig.suptitle(f'{arch_label(arch)} optimization history', fontsize=24, y=0.97)
        fig.text(
            0.5, 0.035,
            r'Sawtooth structure follows cosine learning-rate warm restarts ($T_0=4000$ updates).',
            ha='center', va='bottom', fontsize=13.5, color='0.35',
        )

        out = QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_small_data_loss_curves.png'
        QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, bbox_inches='tight', dpi=300)
        plt.show()
        print('wrote', out)
        return out


small_data_loss_paths = {}
for arch in SANITY_ARCHES:
    display(Markdown(f'### {arch_label(arch)} loss curves for $2^6$--$2^{{10}}$'))
    small_data_loss_paths[arch] = plot_small_data_loss_curves(arch=arch)


## DiT-L16 validity audit

This audit separates **optimization**, **novelty**, and **physical fidelity**. A high q95 nearest-neighbor score only says that a generated map is not unusually close to the training set in that embedding. It does not show that the generated field follows the target distribution.


In [ ]:
def metric_value_at_n(df: pd.DataFrame, arch: str, n: int, column: str) -> float:
    if df.empty or column not in df.columns:
        return np.nan
    tmp = ensure_arch_columns(df)
    sub = tmp[(tmp['arch'].astype(str) == arch) & (tmp['dataset_size'].astype(float) == float(n))]
    return float(sub.iloc[0][column]) if not sub.empty and pd.notna(sub.iloc[0][column]) else np.nan


def load_yaml_if_exists(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    with path.open() as handle:
        return yaml.safe_load(handle) or {}


def build_l16_validity_audit() -> pd.DataFrame:
    arch = 'dit_l16'
    fidelity_path = TABLE_DIR / 'nf_generalize_fig2_dit_l16_fidelity_summary.csv'
    fidelity = read_csv_if_exists(fidelity_path)
    l16_manifest = manifest_df[
        (manifest_df['arch'].astype(str) == arch)
        & manifest_df['dataset_tag'].astype(str).isin(SANITY_TAGS)
    ].copy().sort_values('dataset_size')
    rows = []

    for _, spec in l16_manifest.iterrows():
        n = int(spec['dataset_size'])
        run_name = str(spec['run_name'])
        config_path = config_path_for(spec)
        config = load_yaml_if_exists(config_path)
        model_cfg = config.get('model', {})
        model_kwargs = model_cfg.get('kwargs', {})
        noise_kwargs = config.get('noise_scheduler', {}).get('kwargs', {})
        train_cfg = config.get('train', {})
        data_cfg = config.get('data', {})
        bundle = loaded.get(run_name, {})
        generated = np.asarray(bundle.get('generated', []))

        configuration_ok = bool(
            model_cfg.get('class') == 'DiTTransformer2DModel'
            and int(model_kwargs.get('num_layers', -1)) == 16
            and int(model_kwargs.get('num_attention_heads', -1)) == 12
            and int(model_kwargs.get('attention_head_dim', -1)) == 64
            and noise_kwargs.get('prediction_type') == 'v_prediction'
            and int(data_cfg.get('constant_label', -1)) == 0
            and int(train_cfg.get('gradient_accumulation_steps', -1)) == 4
        )
        sample_ok = bool(generated.ndim == 4 and len(generated) == 512 and np.isfinite(generated).all())

        loss_sub = loss_df[
            (loss_df['arch'].astype(str) == arch)
            & (loss_df['dataset_size'].astype(float) == float(n))
        ] if 'loss_df' in globals() and not loss_df.empty else pd.DataFrame()
        final_loss = float(loss_sub.iloc[0]['final_epoch_loss']) if not loss_sub.empty else np.nan
        best_loss = float(loss_sub.iloc[0]['best_epoch_loss']) if not loss_sub.empty else np.nan

        fidelity_sub = fidelity[fidelity['dataset_size'].astype(float) == float(n)] if not fidelity.empty else pd.DataFrame()
        pk_median = float(fidelity_sub.iloc[0]['pk_ratio_median']) if not fidelity_sub.empty else np.nan
        pk_max_deviation = float(fidelity_sub.iloc[0]['max_abs_pk_ratio_minus_1']) if not fidelity_sub.empty else np.nan
        pca_q95 = metric_value_at_n(pca_metrics, arch, n, 'gen_gl_q95')
        sscd_q95 = metric_value_at_n(sscd_metrics, arch, n, 'gen_gl_q95')
        novelty_score = float(np.nanmax([pca_q95, sscd_q95])) if np.isfinite([pca_q95, sscd_q95]).any() else np.nan

        if np.isfinite(pk_max_deviation) and pk_max_deviation > 0.5:
            status = 'novel_but_physically_invalid' if novelty_score >= 0.5 else 'physically_invalid'
        elif np.isfinite(pk_max_deviation) and pk_max_deviation > 0.25:
            status = 'fidelity_caution'
        else:
            status = 'fidelity_consistent'

        rows.append({
            'dataset_size': n,
            'configuration_ok': configuration_ok,
            'sample_ok': sample_ok,
            'final_epoch_loss': final_loss,
            'best_epoch_loss': best_loss,
            'pca_gen_gl_q95': pca_q95,
            'sscd_gen_gl_q95': sscd_q95,
            'pca_sscd_gap': abs(pca_q95 - sscd_q95),
            'pk_ratio_median': pk_median,
            'max_abs_pk_ratio_minus_1': pk_max_deviation,
            'interpretation': status,
        })

    audit_df = pd.DataFrame(rows)
    display(audit_df.style.format({
        'final_epoch_loss': '{:.3g}',
        'best_epoch_loss': '{:.3g}',
        'pca_gen_gl_q95': '{:.3f}',
        'sscd_gen_gl_q95': '{:.3f}',
        'pca_sscd_gap': '{:.3f}',
        'pk_ratio_median': '{:.3f}',
        'max_abs_pk_ratio_minus_1': '{:.3f}',
    }))
    out = TABLE_DIR / 'nf_generalize_fig2_dit_l16_validity_audit.csv'
    audit_df.to_csv(out, index=False)
    print('wrote', out)
    return audit_df


l16_validity_audit = build_l16_validity_audit()


### What the audit means

- The L16 runs have the intended 16-layer architecture, null class label, v-prediction scheduler, accumulation setting, and finite 512-sample outputs. That makes a simple routing or wrong-file bug unlikely.
- The denoising loss decreases cleanly and is often lower than for L8/L12. The optimizer is fitting its training objective; low diffusion loss is not a sufficient model-selection metric.
- At small $N_{2D}$, L16 can receive a high nearest-neighbor novelty score while its $P(k)$ is wrong by factors of several. Those cases are **novel but physically invalid**, not early generalization.
- The most likely interpretation is that the deeper model is more data- and hyperparameter-sensitive. Confirm it with another seed and a learning-rate/regularization ablation before making a capacity claim.


## DiT Generalization Curves

The q95 score measures nearest-neighbor novelty relative to a real-data baseline. A score near zero means generated fields are unusually close to training slices. **High score means unlike the training set; it does not guarantee physical fidelity.** Interpret these curves together with the image, one-point, and $P(k)$ checks above.


In [ ]:
def format_power_ticks(ax, values):
    vals = sorted({int(v) for v in values if pd.notna(v) and v > 0})
    if not vals:
        return
    ax.set_xscale('log', base=2)
    ax.set_xticks(vals)
    ax.set_xticklabels([rf'$2^{{{int(round(math.log2(v)))}}}$' for v in vals])


def plot_dit_generalization_curves(metrics_by_feature: dict[str, pd.DataFrame], quantile: str = 'q95') -> Path | None:
    gl_col = f'gen_gl_{quantile}'
    fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.3), sharey=True, constrained_layout=True)
    plotted = False

    for ax, (feature_name, df) in zip(axes, metrics_by_feature.items()):
        all_x = []
        if df.empty or gl_col not in df.columns or 'dataset_size' not in df.columns:
            ax.set_visible(False)
            continue
        df = ensure_arch_columns(df)
        for arch in DIT_ARCH_ORDER:
            sub = df[df['arch'].astype(str) == arch].dropna(subset=['dataset_size', gl_col]).sort_values('dataset_size')
            if sub.empty:
                continue
            all_x.extend(sub['dataset_size'].astype(float).tolist())
            ax.plot(
                sub['dataset_size'], sub[gl_col],
                marker=DIT_ARCH_MARKERS.get(arch, 'o'), ms=8, lw=3,
                color=DIT_ARCH_COLORS.get(arch, '0.2'), label=arch_label(arch),
            )
            plotted = True
        format_power_ticks(ax, all_x)
        ax.axhline(0.5, color='0.35', lw=1.5, ls=':', label='0.5 marker')
        ax.set_ylim(-0.04, 1.04)
        ax.set_xlabel(r'Training set size $N_{2D}$')
        ax.set_title(f'{feature_name}: DiT depth sweep at 200k updates ({quantile})')
        ax.grid(True, alpha=0.22)
        ax.legend(frameon=False, loc='lower right')
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
    axes[0].set_ylabel('Generalization score')

    if not plotted:
        display(Markdown(f'No `{gl_col}` columns found to plot.'))
        plt.close(fig)
        return None

    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_depth_gl_curves_{quantile}.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

combined_curve = plot_dit_generalization_curves({'PCA': pca_metrics, 'SSCD': sscd_metrics}, quantile='q95')

## Transition Summary

`N50` is the interpolated training-set size where the generalization score crosses 0.5. The table below uses the original fixed 200k-update sweep. This is a compact comparison, but it should be read with the full curve and the later L16 continuation experiment because a single midpoint can hide optimization sensitivity, changes in slope, or physically invalid samples.


In [ ]:
def interpolate_crossing(df: pd.DataFrame, ycol: str, threshold: float = 0.5) -> dict[str, Any]:
    if df.empty or ycol not in df.columns or 'dataset_size' not in df.columns:
        return {'status': 'missing', 'n_cross': np.nan, 'log2_n_cross': np.nan}
    sub = df[['dataset_size', ycol]].dropna().sort_values('dataset_size')
    sub = sub[sub['dataset_size'] > 0]
    if sub.empty:
        return {'status': 'missing', 'n_cross': np.nan, 'log2_n_cross': np.nan}

    x = np.log2(sub['dataset_size'].astype(float).to_numpy())
    y = sub[ycol].astype(float).to_numpy()
    if y[0] >= threshold:
        return {'status': 'left_censored', 'n_cross': 2 ** x[0], 'log2_n_cross': x[0]}
    if y[-1] < threshold:
        return {'status': 'right_censored', 'n_cross': 2 ** x[-1], 'log2_n_cross': x[-1]}

    for i in range(len(y) - 1):
        y0, y1 = y[i], y[i + 1]
        if (y0 <= threshold <= y1) or (y1 <= threshold <= y0):
            if y1 == y0:
                xc = x[i]
            else:
                frac = (threshold - y0) / (y1 - y0)
                xc = x[i] + frac * (x[i + 1] - x[i])
            return {'status': 'interpolated', 'n_cross': 2 ** xc, 'log2_n_cross': xc}
    return {'status': 'not_found', 'n_cross': np.nan, 'log2_n_cross': np.nan}


rows = []
for feature_name, df in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
    df = ensure_arch_columns(df)
    for arch in DIT_ARCH_ORDER:
        sub = df[df['arch'].astype(str) == arch]
        for q in ['q90', 'q95', 'q99']:
            col = f'gen_gl_{q}'
            result = interpolate_crossing(sub, col, threshold=0.5)
            rows.append({
                'feature': feature_name,
                'arch': arch,
                'arch_label': arch_label(arch),
                'score_col': col,
                'threshold': 0.5,
                **result,
            })

transition_df = pd.DataFrame(rows)
display(transition_df.sort_values(['feature', 'arch', 'score_col']))

if len(transition_df):
    out = TABLE_DIR / 'nf_generalize_fig2_dit_transition_summary.csv'
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    transition_df.to_csv(out, index=False)
    print('wrote', out)

## Complete 200k-update depth comparison

This is the main architecture comparison. Every DiT depth uses the same 200k optimizer-update budget and is shown over the complete $2^6$ to $2^{15}$ data-size sweep. The UNet curves are fixed-budget references. The later DiT-L16 continuation covers only $2^6$ to $2^{10}$, so it is kept separate and is not used as a depth-scaling curve.


In [ ]:
UNET_RESULTS_DIR = PROJECT_DIR / 'results' / 'nf_generalize_fig2'
UNET_TABLE_DIR = UNET_RESULTS_DIR / 'tables'

unet_pca = add_generalization_columns(read_csv_if_exists(UNET_TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_metrics.csv'))
unet_sscd = add_generalization_columns(read_csv_if_exists(UNET_TABLE_DIR / 'nf_generalize_fig2_sscd_full_nn_metrics.csv'))


def plot_dit_vs_unet_combined(
    dit_by_feature: dict[str, pd.DataFrame],
    unet_by_feature: dict[str, pd.DataFrame],
    quantile: str = 'q95',
) -> Path | None:
    gl_col = f'gen_gl_{quantile}'
    feature_order = [('PCA', 'PCA embedding'), ('SSCD', 'SSCD embedding')]
    if not any(not df.empty and gl_col in df.columns for df in dit_by_feature.values()):
        display(Markdown(f'No DiT `{gl_col}` data available.'))
        return None

    dit_colors = {'dit_l8': '#009E73', 'dit_base': '#0072B2', 'dit_l16': '#CC79A7'}
    unet_colors = {'u64': '#B8B8B8', 'u128': '#858585', 'u256': '#505050'}
    unet_dashes = {'u64': (0, (5, 3)), 'u128': (0, (2, 2)), 'u256': (0, (8, 3))}

    with plt.rc_context({
        'font.family': 'serif',
        'font.serif': ['DejaVu Serif'],
        'mathtext.fontset': 'dejavuserif',
        'font.size': 16,
        'axes.labelsize': 20,
        'axes.titlesize': 22,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16,
        'legend.fontsize': 14,
    }):
        fig, axes = plt.subplots(1, 2, figsize=(17.0, 6.5), sharey=True)
        fig.subplots_adjust(left=0.075, right=0.985, bottom=0.15, top=0.70, wspace=0.08)

        for ax, (feature_key, panel_title) in zip(axes, feature_order):
            dit_df = ensure_arch_columns(dit_by_feature.get(feature_key, pd.DataFrame()))
            unet_df = ensure_arch_columns(unet_by_feature.get(feature_key, pd.DataFrame()))
            all_x = []

            if not unet_df.empty and gl_col in unet_df.columns:
                for arch in UNET_ARCH_ORDER:
                    sub = unet_df[unet_df['arch'].astype(str) == arch].dropna(
                        subset=['dataset_size', gl_col]
                    ).sort_values('dataset_size')
                    if sub.empty:
                        continue
                    all_x.extend(sub['dataset_size'].astype(float).tolist())
                    ax.plot(
                        sub['dataset_size'], sub[gl_col],
                        color=unet_colors[arch], linestyle=unet_dashes[arch],
                        marker=UNET_ARCH_MARKERS[arch], markerfacecolor='white',
                        markeredgewidth=1.4, lw=1.9, ms=7.0, alpha=0.95,
                        zorder=1,
                    )

            if not dit_df.empty and gl_col in dit_df.columns:
                for arch in DIT_ARCH_ORDER:
                    sub = dit_df[dit_df['arch'].astype(str) == arch].dropna(
                        subset=['dataset_size', gl_col]
                    ).sort_values('dataset_size')
                    if sub.empty:
                        continue
                    all_x.extend(sub['dataset_size'].astype(float).tolist())
                    ax.plot(
                        sub['dataset_size'], sub[gl_col],
                        color=dit_colors[arch], marker=DIT_ARCH_MARKERS[arch],
                        markeredgecolor='white', markeredgewidth=0.8,
                        lw=3.5, ms=9.5, zorder=3,
                    )

            ax.axhline(0.5, color='0.35', lw=1.4, ls=':', zorder=0)
            format_power_ticks(ax, all_x)
            ax.set_ylim(-0.035, 1.035)
            ax.set_xlabel(r'Training images $N_{2D}$', labelpad=8)
            ax.set_title(panel_title, pad=12, fontweight='semibold')
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(width=1.1, length=5)

        axes[0].set_ylabel('q95 novelty score', labelpad=8)
        axes[1].tick_params(labelleft=False)
        axes[0].text(66, 0.515, '0.5 reference', fontsize=12.5, color='0.35', va='bottom')

        legend_handles = []
        legend_labels = []
        for arch in UNET_ARCH_ORDER:
            legend_handles.append(Line2D(
                [0], [0], color=unet_colors[arch], linestyle=unet_dashes[arch],
                marker=UNET_ARCH_MARKERS[arch], markerfacecolor='white',
                markeredgewidth=1.3, lw=1.9, ms=7.0,
            ))
            legend_labels.append(arch_label(arch))
        for arch in DIT_ARCH_ORDER:
            legend_handles.append(Line2D(
                [0], [0], color=dit_colors[arch], marker=DIT_ARCH_MARKERS[arch],
                markeredgecolor='white', markeredgewidth=0.8, lw=3.5, ms=9.0,
            ))
            legend_labels.append(arch_label(arch))

        fig.suptitle('DiT depth sweep at fixed 200k updates', fontsize=27, y=0.97, fontweight='semibold')
        fig.text(
            0.5, 0.895,
            'UNet references and all DiT depths use 200k optimizer updates. High novelty does not guarantee physical fidelity.',
            ha='center', va='center', fontsize=16.5, color='0.28',
        )
        fig.legend(
            legend_handles, legend_labels,
            loc='upper center', bbox_to_anchor=(0.5, 0.835),
            ncol=6, frameon=False, handlelength=2.2, columnspacing=1.5,
        )

        out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_depth_vs_unet_pca_sscd_{quantile}.png'
        QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, bbox_inches='tight', dpi=300)
        plt.show()
        print('wrote', out)
        return out


combined_dit_unet_comparison = plot_dit_vs_unet_combined(
    {'PCA': pca_metrics, 'SSCD': sscd_metrics},
    {'PCA': unet_pca, 'SSCD': unet_sscd},
    quantile='q95',
)


## Fixed-Budget Capacity Check: DiT Depth vs UNet Width

This exploratory table and plot use the original 200k-update checkpoints for every DiT depth. They ask how many 2D images are needed before the q95 novelty score crosses 0.5. The x-axis is trainable parameter count, and the y-axis is the interpolated crossing size `N50`. Do not interpret the DiT-L16 point as a capacity law when its small-data samples fail physical-statistics checks.

In [ ]:
# Parameter counts.
# UNet values are exact from the existing sweep. DiT values are approximate for width 768,
# patch 8, and depths 8/12/16; replace with print_model_param_count.py values if needed.
MODEL_CAPACITY = pd.DataFrame([
    {'model': 'UNet-64',        'family': 'UNet', 'arch_key': 'u64',      'model_params': 26_621_057,  'source': 'exact'},
    {'model': 'UNet-128',       'family': 'UNet', 'arch_key': 'u128',     'model_params': 140_539_521, 'source': 'exact'},
    {'model': 'UNet-256',       'family': 'UNet', 'arch_key': 'u256',     'model_params': 196_059_905, 'source': 'exact'},
    {'model': 'DiT-L8',         'family': 'DiT',  'arch_key': 'dit_l8',   'model_params': 95_000_000,  'source': 'approx'},
    {'model': 'DiT-L12 / base', 'family': 'DiT',  'arch_key': 'dit_base', 'model_params': 138_290_000, 'source': 'approx'},
    {'model': 'DiT-L16',        'family': 'DiT',  'arch_key': 'dit_l16',  'model_params': 182_000_000, 'source': 'approx'},
])
MODEL_CAPACITY['model_params_m'] = MODEL_CAPACITY['model_params'] / 1e6


def n50_for_model(feature_name: str, model: str, family: str, model_params: int, df: pd.DataFrame, q: str = 'q95') -> dict[str, Any]:
    col = f'gen_gl_{q}'
    cross = interpolate_crossing(df, col, threshold=0.5)
    return {
        'feature': feature_name,
        'model': model,
        'family': family,
        'model_params': model_params,
        'model_params_m': model_params / 1e6,
        'score_col': col,
        **cross,
    }


def capacity_transition_table(q: str = 'q95') -> pd.DataFrame:
    rows = []
    feature_frames = {
        'PCA': (pca_metrics, unet_pca),
        'SSCD': (sscd_metrics, unet_sscd),
    }
    for feature_name, (dit_df, unet_df) in feature_frames.items():
        dit_tmp = ensure_arch_columns(dit_df)
        for arch in DIT_ARCH_ORDER:
            cap = MODEL_CAPACITY[MODEL_CAPACITY['arch_key'] == arch]
            if cap.empty:
                continue
            cap = cap.iloc[0]
            sub = dit_tmp[dit_tmp['arch'].astype(str) == arch]
            rows.append(n50_for_model(feature_name, cap['model'], cap['family'], int(cap['model_params']), sub, q=q))

        if unet_df.empty:
            continue
        unet_tmp = ensure_arch_columns(unet_df)
        for arch in UNET_ARCH_ORDER:
            cap = MODEL_CAPACITY[MODEL_CAPACITY['arch_key'] == arch].iloc[0]
            sub = unet_tmp[unet_tmp['arch'].astype(str) == arch]
            rows.append(n50_for_model(feature_name, cap['model'], cap['family'], int(cap['model_params']), sub, q=q))
    out = pd.DataFrame(rows)
    return out.sort_values(['feature', 'family', 'model_params']).reset_index(drop=True)


capacity_n50 = capacity_transition_table(q='q95')
display(capacity_n50[['feature', 'model', 'family', 'model_params_m', 'status', 'n_cross', 'log2_n_cross']])

# Pairwise comparison against nearby UNet capacities.
ratio_rows = []
for feature_name in ['PCA', 'SSCD']:
    sub = capacity_n50[capacity_n50['feature'] == feature_name].set_index('model')
    for dit_model in ['DiT-L8', 'DiT-L12 / base', 'DiT-L16']:
        if dit_model not in sub.index:
            continue
        for baseline in ['UNet-128', 'UNet-256']:
            if baseline not in sub.index:
                continue
            ratio_rows.append({
                'feature': feature_name,
                'comparison': f'{dit_model} / {baseline}',
                'param_ratio': sub.loc[dit_model, 'model_params'] / sub.loc[baseline, 'model_params'],
                'n50_ratio': sub.loc[dit_model, 'n_cross'] / sub.loc[baseline, 'n_cross'],
                'dit_log2_n50': sub.loc[dit_model, 'log2_n_cross'],
                'baseline_log2_n50': sub.loc[baseline, 'log2_n_cross'],
            })
capacity_ratios = pd.DataFrame(ratio_rows)
display(capacity_ratios)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.3), sharey=True, constrained_layout=True)
for ax, feature_name in zip(axes, ['PCA', 'SSCD']):
    sub = capacity_n50[(capacity_n50['feature'] == feature_name) & capacity_n50['n_cross'].notna()].copy()
    if sub.empty:
        ax.set_visible(False)
        continue
    unet = sub[sub['family'] == 'UNet'].sort_values('model_params')
    dit = sub[sub['family'] == 'DiT'].sort_values('model_params')
    if len(unet):
        ax.plot(unet['model_params'], unet['n_cross'], color='#0072B2', marker='o', lw=2.4, ms=8, label='UNet family')
        for _, row in unet.iterrows():
            ax.annotate(row['model'].replace('UNet-', 'U'), (row['model_params'], row['n_cross']), xytext=(5, 4), textcoords='offset points', fontsize=10)
    if len(dit):
        ax.plot(dit['model_params'], dit['n_cross'], color='black', marker='D', lw=2.4, ms=8, label='DiT depth')
        for _, row in dit.iterrows():
            ax.annotate(row['model'].replace('DiT-', 'D'), (row['model_params'], row['n_cross']), xytext=(6, -12), textcoords='offset points', fontsize=10, color='black')
    ax.set_xscale('log')
    ax.set_yscale('log', base=2)
    ax.set_xlabel('Trainable parameters')
    ax.set_title(f'{feature_name}: q95 transition at 200k updates')
    ax.grid(True, alpha=0.22)
    ax.legend(frameon=False, loc='upper left')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
axes[0].set_ylabel(r'$N_{50}$ training images')

out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_depth_capacity_n50_q95.png'
fig.savefig(out, bbox_inches='tight')
plt.show()
print('wrote', out)

if not capacity_ratios.empty:
    lines = ['### Fixed-200k capacity interpretation']
    for _, row in capacity_ratios.iterrows():
        if not np.isfinite(row['n50_ratio']):
            continue
        lines.append(
            f"- {row['feature']}: {row['comparison']} has {row['param_ratio']:.2f}x parameters "
            f"and {row['n50_ratio']:.2f}x the q95 N50."
        )
    lines.append('- This is an exploratory fixed-budget diagnostic, not a universal scaling law. The DiT-L16 point is not reliable capacity evidence while some high-novelty samples are physically invalid and its transition changes with continued optimization.')
    display(Markdown('\n'.join(lines)))

## Existing Quickcheck Figures and Saved Diagnostics


In [ ]:
def show_existing_figure(path: Path, title: str, width: int = 950) -> None:
    display(Markdown(f'### {title}'))
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f'Missing: `{rel(path)}`'))

show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_training_curves.png', 'DiT training curves')
for arch in DIT_ARCH_ORDER:
    show_existing_figure(QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_generated_image_grid.png', f'{arch_label(arch)} generated image grid')
    show_existing_figure(QUICKCHECK_DIR / f'nf_generalize_fig2_{arch}_detail_onepoint_pk.png', f'{arch_label(arch)} one-point and P(k) fidelity')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_depth_gl_curves_q95.png', 'DiT depth GL curves at fixed 200k updates')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_depth_vs_unet_pca_sscd_q95.png', 'DiT depth vs UNet at fixed 200k updates')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_depth_capacity_n50_q95.png', 'Exploratory fixed-200k capacity diagnostic')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_paper_style_gl_curves.png', 'PCA paper-style GL curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_paper_style_gl_curves.png', 'SSCD paper-style GL curves')

## Sample File Sanity Check

This inspects array keys and shapes without doing the expensive nearest-neighbor analysis again.


In [ ]:
def inspect_npz(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {'exists': False}
    with np.load(path) as data:
        keys = list(data.files)
        first = keys[0] if keys else None
        arr = data[first] if first else None
        return {
            'exists': True,
            'keys': ', '.join(keys[:8]),
            'first_key': first,
            'shape': tuple(arr.shape) if arr is not None else None,
            'dtype': str(arr.dtype) if arr is not None else None,
            'size_mb': path.stat().st_size / 1024**2,
        }

if manifest_df.empty:
    display(Markdown('No manifest available, so sample files were not inspected.'))
else:
    rows = []
    for _, row in manifest_df.sort_values(['arch', 'dataset_size']).iterrows():
        path = row['sample_path_resolved']
        info = inspect_npz(path)
        rows.append({
            'arch_label': row.get('arch_label'),
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': row.get('dataset_size'),
            'path': rel(path),
            **info,
        })
    sample_inspect_df = pd.DataFrame(rows)
    display(sample_inspect_df)

## Controlled DiT-L16 continuation: low-data diagnostic, not a scaling curve

This section inspects DiT-L16 at $N_{2D}=2^6,\ldots,2^{10}$ across the original 200k sample and the saved 225k, 250k, 275k, and 300k samples. It does **not** cover $2^{11}$ to $2^{15}$, so it cannot establish whether the full depth transition moves right with additional optimization.

### Resume-state warning

The currently saved 225k to 300k samples were generated by an older class-safe resume loader that restored the DiT weights but recreated the optimizer and learning-rate scheduler state at each 25k stage. AdamW moments and scheduler progress were therefore reset. These files are useful for diagnosing failure modes, but they are not a controlled longer-training trajectory and are excluded from the scaling comparison above.

The corrected loader restores `optimizer.pkl`, `lr_scheduler.pkl`, and `noise_scheduler.pkl` and fails if those files are missing. Rerun the continuation from the original 200k checkpoints before using later checkpoints to make a training-dynamics claim.

Every physical-statistics comparison below reloads the **complete configured training reference**. A run is rejected if the number of real slices does not equal its manifest `dataset_size`. Novelty must be read together with generated images, the one-point PDF, and $P(k)$: off-distribution noise can be far from every training image and therefore receive a high novelty score.


In [ ]:
CONTINUE_MANIFEST_PATH = PROJECT_DIR / 'local' / 'nf_generalize_fig2_dit_l16_continue' / 'manifest.json'
CONTINUE_ANALYSIS_MANIFEST_PATH = CONTINUE_MANIFEST_PATH.parent / 'analysis_manifest.json'
CONTINUE_CHECKPOINTS = [
    (200, 'dpm50'),
    (225, 'dpm50_cont_225k'),
    (250, 'dpm50_cont_250k'),
    (275, 'dpm50_cont_275k'),
    (300, 'dpm50_cont_300k'),
]
CONTINUE_TAGS = ['d2p06', 'd2p07', 'd2p08', 'd2p09', 'd2p10']
CONTINUE_DETAIL_TAG = os.environ.get('DIT_CONTINUE_DETAIL_TAG', 'd2p08')

continue_manifest_obj = read_json(CONTINUE_MANIFEST_PATH)
continue_analysis_obj = read_json(CONTINUE_ANALYSIS_MANIFEST_PATH)
if continue_manifest_obj is None:
    display(Markdown(f'Continuation manifest not created yet: `{rel(CONTINUE_MANIFEST_PATH)}`'))
    continue_manifest_df = pd.DataFrame()
else:
    continue_manifest_df = pd.DataFrame(continue_manifest_obj)
    display(continue_manifest_df[[
        'continue_stage', 'run_name', 'dataset_tag', 'dataset_size',
        'target_total_updates', 'expected_checkpoint', 'sample_label',
    ]].sort_values(['continue_stage', 'dataset_size']))

if continue_analysis_obj is None:
    continue_base_df = pd.DataFrame()
else:
    continue_base_df = ensure_arch_columns(pd.DataFrame(continue_analysis_obj))
    continue_base_df = continue_base_df[
        (continue_base_df['arch'].astype(str) == 'dit_l16')
        & continue_base_df['dataset_tag'].isin(CONTINUE_TAGS)
    ].sort_values('dataset_size')


def continuation_sample_path(run_name: str, label: str) -> Path:
    return SAMPLE_DIR / f'{run_name}_seed{SEED}_{label}.npz'


def continuation_table_path(feature: str, updates_k: int) -> Path:
    return TABLE_DIR / f'nf_generalize_fig2_dit_l16_cont_{updates_k}k_{feature.lower()}_full_nn_metrics.csv'


sample_audit_rows = []
for _, row in continue_base_df.iterrows():
    for updates_k, label in CONTINUE_CHECKPOINTS:
        sample_path = continuation_sample_path(str(row['run_name']), label)
        sample_audit_rows.append({
            'dataset_tag': row['dataset_tag'],
            'dataset_size': int(row['dataset_size']),
            'updates_k': updates_k,
            'sample_label': label,
            'sample_exists': sample_path.exists(),
            'sample_path': rel(sample_path),
        })
continuation_sample_audit_df = pd.DataFrame(sample_audit_rows)
display(continuation_sample_audit_df)


In [ ]:
continuation_fidelity_rows = []
continuation_curve_cache = {}
continuation_image_cache = {}

if continue_base_df.empty or not SIMDIFF_EVAL_AVAILABLE:
    display(Markdown('Continuation physical-statistics check is waiting for its manifest or `simdiff_eval`.'))
else:
    for _, row in continue_base_df.iterrows():
        run_name = str(row['run_name'])
        dataset_tag = str(row['dataset_tag'])
        dataset_size = int(row['dataset_size'])
        cfg_path = config_path_for(row)
        real = as_nchw(load_real_from_config(cfg_path, max_raw_samples=None))
        if len(real) != dataset_size:
            raise RuntimeError(
                'FULL TRAINING REFERENCE MISMATCH: '
                f'{run_name} loaded {len(real)} real slices from {cfg_path}; expected {dataset_size}.'
            )

        real_hist = field_histogram(real, bins=140)
        edges = np.asarray(real_hist['bin_edges'], dtype=float)
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        real_density = np.asarray(real_hist['hist'], dtype=float)
        pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
        mean_pk_real = np.clip(np.nanmean(pk_real, axis=0), 1e-30, None)

        for updates_k, label in CONTINUE_CHECKPOINTS:
            sample_path = continuation_sample_path(run_name, label)
            if not sample_path.exists():
                continue
            generated = evenly_limit(load_npz_array(sample_path), MAX_GENERATED)
            generated_density, _ = np.histogram(generated.ravel(), bins=edges, density=True)
            hist_l1 = float(np.sum(np.abs(generated_density - real_density) * widths))
            pk_generated, _ = batch_power_spectra(generated, nbins=PK_NBINS)
            ratio = np.nanmean(pk_generated, axis=0) / mean_pk_real
            finite = np.isfinite(ratio) & (ratio > 0)
            pk_log10_mae = float(np.mean(np.abs(np.log10(ratio[finite])))) if finite.any() else np.nan
            thirds = np.array_split(np.where(finite)[0], 3) if finite.any() else [[], [], []]
            band_ratio = [float(np.nanmean(ratio[idx])) if len(idx) else np.nan for idx in thirds]

            resolved_checkpoint = 'missing (legacy 200k sample)'
            with np.load(sample_path, allow_pickle=False) as data:
                if 'resolved_checkpoint' in data.files:
                    resolved_checkpoint = str(np.asarray(data['resolved_checkpoint']).item())

            continuation_fidelity_rows.append({
                'run_name': run_name,
                'dataset_tag': dataset_tag,
                'dataset_size': dataset_size,
                'updates_k': updates_k,
                'sample_label': label,
                'n_real': len(real),
                'n_generated': len(generated),
                'real_reference_kind': 'complete configured training set',
                'real_config_path': str(cfg_path),
                'resolved_checkpoint': resolved_checkpoint,
                'hist_l1': hist_l1,
                'pk_log10_mae': pk_log10_mae,
                'pk_ratio_low_k': band_ratio[0],
                'pk_ratio_mid_k': band_ratio[1],
                'pk_ratio_high_k': band_ratio[2],
            })
            continuation_curve_cache[(dataset_tag, updates_k)] = {
                'centers': centers.copy(),
                'real_density': real_density.copy(),
                'generated_density': generated_density.copy(),
                'kbins': np.asarray(kbins).copy(),
                'pk_ratio': np.asarray(ratio).copy(),
            }
            continuation_image_cache[(dataset_tag, updates_k)] = generated[:4, 0].copy()

continuation_fidelity_df = pd.DataFrame(continuation_fidelity_rows)
if continuation_fidelity_df.empty:
    display(Markdown('No continuation samples are available yet. Rerun this section after the staged sampler jobs finish.'))
else:
    display(continuation_fidelity_df.sort_values(['dataset_size', 'updates_k']))
    fidelity_out = TABLE_DIR / 'nf_generalize_fig2_dit_l16_continuation_fidelity.csv'
    continuation_fidelity_df.to_csv(fidelity_out, index=False)
    print('wrote', fidelity_out)

    fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), constrained_layout=True)
    colors = plt.cm.viridis(np.linspace(0.08, 0.90, len(CONTINUE_TAGS)))
    for color, tag in zip(colors, CONTINUE_TAGS):
        sub = continuation_fidelity_df[continuation_fidelity_df['dataset_tag'] == tag].sort_values('updates_k')
        if sub.empty:
            continue
        label = dataset_size_label(int(sub['dataset_size'].iloc[0]))
        axes[0].plot(sub['updates_k'], sub['hist_l1'], marker='o', lw=2.5, color=color, label=label)
        axes[1].plot(sub['updates_k'], sub['pk_log10_mae'], marker='o', lw=2.5, color=color, label=label)
    axes[0].set_title('One-point PDF error')
    axes[0].set_ylabel(r'$L_1$ distance (lower is better)')
    axes[1].set_title('Power-spectrum error')
    axes[1].set_ylabel(r'mean $|\log_{10}(P_{gen}/P_{real})|$ (lower is better)')
    for ax in axes:
        ax.set_xlabel('Optimizer updates (thousands)')
        ax.set_xticks([x for x, _ in CONTINUE_CHECKPOINTS])
        ax.grid(alpha=0.18)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    axes[1].legend(title=r'$N_{2D}$', frameon=False, ncol=1)
    fig.suptitle('Does additional optimization repair DiT-L16 small-data fidelity?', y=1.03)
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_continuation_fidelity_trajectories.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)


In [ ]:
def four_sample_montage(images: np.ndarray) -> np.ndarray:
    images = np.asarray(images)
    if images.ndim != 3 or len(images) < 4:
        raise ValueError(f'Expected at least four 2D samples, found shape {images.shape}.')
    return np.block([[images[0], images[1]], [images[2], images[3]]])


if continuation_image_cache:
    available_updates = [
        updates_k
        for updates_k, _ in CONTINUE_CHECKPOINTS
        if any((tag, updates_k) in continuation_image_cache for tag in CONTINUE_TAGS)
    ]
    available_tags = [
        tag
        for tag in CONTINUE_TAGS
        if any((tag, updates_k) in continuation_image_cache for updates_k in available_updates)
    ]
    values = np.concatenate([
        continuation_image_cache[(tag, updates_k)].ravel()
        for updates_k in available_updates
        for tag in available_tags
        if (tag, updates_k) in continuation_image_cache
    ])
    vmin, vmax = np.quantile(values, [0.005, 0.995])
    fig, axes = plt.subplots(
        len(available_updates),
        len(available_tags),
        figsize=(3.2 * len(available_tags), 3.0 * len(available_updates)),
        squeeze=False,
        constrained_layout=True,
    )
    for row_index, updates_k in enumerate(available_updates):
        for column_index, tag in enumerate(available_tags):
            ax = axes[row_index, column_index]
            images = continuation_image_cache.get((tag, updates_k))
            if images is not None:
                ax.imshow(four_sample_montage(images), cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_xticks([])
            ax.set_yticks([])
            if row_index == 0:
                ax.set_title(dataset_size_label(dataset_size_from_tag(tag)), fontsize=15)
            if column_index == 0:
                ax.set_ylabel(f'{updates_k}k', fontsize=14, fontweight='semibold')
    fig.suptitle(
        'DiT-L16 low-data checkpoint audit: four generated maps per checkpoint',
        y=1.02,
        fontsize=20,
    )
    out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_continuation_image_grid.png'
    fig.savefig(out, bbox_inches='tight', dpi=240)
    plt.show()
    print('wrote', out)

if continuation_curve_cache:
    detail_updates = [
        updates_k
        for updates_k, _ in CONTINUE_CHECKPOINTS
        if (CONTINUE_DETAIL_TAG, updates_k) in continuation_curve_cache
    ]
    if detail_updates:
        cmap = plt.cm.plasma(np.linspace(0.12, 0.88, len(detail_updates)))
        first = continuation_curve_cache[(CONTINUE_DETAIL_TAG, detail_updates[0])]
        fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), constrained_layout=True)
        axes[0].plot(
            first['centers'],
            first['real_density'],
            color='black',
            lw=2.7,
            label='complete training reference',
        )
        axes[1].axhline(1.0, color='black', ls='--', lw=1.5, label='ideal ratio')
        for color, updates_k in zip(cmap, detail_updates):
            curves = continuation_curve_cache[(CONTINUE_DETAIL_TAG, updates_k)]
            axes[0].plot(
                curves['centers'],
                curves['generated_density'],
                color=color,
                lw=2.1,
                label=f'{updates_k}k',
            )
            axes[1].plot(
                curves['kbins'],
                curves['pk_ratio'],
                color=color,
                marker='o',
                ms=4.0,
                lw=2.0,
                label=f'{updates_k}k',
            )
        axes[0].set_yscale('log')
        axes[0].set_xlabel('Normalized field value')
        axes[0].set_ylabel('Pixel PDF')
        axes[0].set_title('One-point distribution')
        axes[1].set_xlabel(r'$k$ bin')
        axes[1].set_ylabel(r'$P_{generated}(k)/P_{real}(k)$')
        axes[1].set_title('Power-spectrum fidelity')
        for ax in axes:
            ax.grid(False)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.legend(frameon=False)
        n_label = dataset_size_label(dataset_size_from_tag(CONTINUE_DETAIL_TAG))
        fig.suptitle(
            f'Legacy state-reset continuation at $N_{{2D}}$ = {n_label}',
            y=1.03,
        )
        out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_l16_{CONTINUE_DETAIL_TAG}_continuation_pdf_pk.png'
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)


In [ ]:
continuation_table_audit_rows = []
novelty_parts = []
for updates_k, _label in CONTINUE_CHECKPOINTS:
    for feature in ('pca', 'sscd'):
        table_path = continuation_table_path(feature, updates_k)
        continuation_table_audit_rows.append({
            'feature': feature.upper(),
            'updates_k': updates_k,
            'exists': table_path.exists(),
            'table_path': rel(table_path),
        })
        if not table_path.exists():
            continue
        table = add_generalization_columns(pd.read_csv(table_path))
        table = ensure_arch_columns(table)
        table = table[table['arch'].astype(str) == 'dit_l16'].copy()
        table['feature'] = feature.upper()
        table['updates_k'] = updates_k
        table['analysis_manifest'] = str(CONTINUE_ANALYSIS_MANIFEST_PATH)
        novelty_parts.append(table)

continuation_table_audit_df = pd.DataFrame(continuation_table_audit_rows)
display(continuation_table_audit_df)

continuation_novelty_df = pd.concat(novelty_parts, ignore_index=True) if novelty_parts else pd.DataFrame()
if continuation_novelty_df.empty:
    display(Markdown('PCA/SSCD continuation tables are not available yet. They are submitted after each exact-checkpoint sampler array.'))
else:
    display(continuation_novelty_df[[
        c for c in ['feature', 'updates_k', 'dataset_tag', 'dataset_size', 'gen_gl_q95', 'sample_path']
        if c in continuation_novelty_df.columns
    ]].sort_values(['feature', 'dataset_size', 'updates_k']))
    available_novelty_updates = sorted(continuation_novelty_df['updates_k'].dropna().astype(int).unique().tolist())
    if 'gen_gl_q95' in continuation_novelty_df.columns and len(available_novelty_updates) >= 2:
        fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True, constrained_layout=True)
        colors = plt.cm.viridis(np.linspace(0.08, 0.90, len(CONTINUE_TAGS)))
        for ax, feature in zip(axes, ('PCA', 'SSCD')):
            feature_df = continuation_novelty_df[continuation_novelty_df['feature'] == feature]
            for color, tag in zip(colors, CONTINUE_TAGS):
                sub = feature_df[feature_df['dataset_tag'] == tag].sort_values('updates_k')
                if sub.empty:
                    continue
                ax.plot(sub['updates_k'], sub['gen_gl_q95'], marker='o', lw=2.5, color=color,
                        label=dataset_size_label(int(sub['dataset_size'].iloc[0])))
            ax.axhline(0.5, color='0.35', ls=':', lw=1.4)
            ax.set_title(feature)
            ax.set_xlabel('Optimizer updates (thousands)')
            ax.set_xticks([x for x, _ in CONTINUE_CHECKPOINTS])
            ax.set_ylim(-0.04, 1.04)
            ax.grid(alpha=0.18)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        axes[0].set_ylabel('q95 novelty score')
        axes[1].legend(title=r'$N_{2D}$', frameon=False)
        fig.suptitle('Does DiT-L16 novelty change with additional optimization?', y=1.03)
        out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_continuation_pca_sscd.png'
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)

        # Main transition view: one curve per checkpoint over training-set size.
        fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0), sharey=True, constrained_layout=True)
        update_colors = plt.cm.plasma(np.linspace(0.12, 0.88, len(available_novelty_updates)))
        dataset_sizes = [dataset_size_from_tag(tag) for tag in CONTINUE_TAGS]
        for ax, feature in zip(axes, ('PCA', 'SSCD')):
            feature_df = continuation_novelty_df[continuation_novelty_df['feature'] == feature]
            for color, updates_k in zip(update_colors, available_novelty_updates):
                sub = feature_df[feature_df['updates_k'] == updates_k].sort_values('dataset_size')
                if sub.empty:
                    continue
                ax.plot(
                    sub['dataset_size'], sub['gen_gl_q95'],
                    marker='o', ms=7.0, lw=2.5, color=color, label=f'{updates_k}k',
                )
            ax.axhline(0.5, color='0.35', ls=':', lw=1.4)
            ax.set_xscale('log', base=2)
            ax.set_xticks(dataset_sizes)
            ax.set_xticklabels([dataset_size_label(n) for n in dataset_sizes])
            ax.set_xlabel(r'Training set size $N_{2D}$')
            ax.set_title(feature)
            ax.set_ylim(-0.04, 1.04)
            ax.grid(alpha=0.18)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        axes[0].set_ylabel('q95 novelty score')
        axes[1].legend(title='Optimizer updates', frameon=False, ncol=1)
        fig.suptitle('Low-data DiT-L16 checkpoint diagnostic (not a scaling curve)', y=1.03)
        out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_continuation_by_data_size.png'
        fig.savefig(out, bbox_inches='tight')
        plt.show()
        print('wrote', out)

        comparison_rows = []
        for feature in ('PCA', 'SSCD'):
            feature_df = continuation_novelty_df[continuation_novelty_df['feature'] == feature]
            early = feature_df[feature_df['updates_k'] == 200][
                ['dataset_tag', 'dataset_size', 'gen_gl_q95']
            ].rename(columns={'gen_gl_q95': 'q95_200k'})
            late = feature_df[feature_df['updates_k'] == 300][
                ['dataset_tag', 'dataset_size', 'gen_gl_q95']
            ].rename(columns={'gen_gl_q95': 'q95_300k'})
            paired = early.merge(late, on=['dataset_tag', 'dataset_size'], how='outer')
            paired['feature'] = feature
            paired['delta_300k_minus_200k'] = paired['q95_300k'] - paired['q95_200k']
            comparison_rows.append(paired)
        l16_200k_vs_300k = pd.concat(comparison_rows, ignore_index=True).sort_values(
            ['feature', 'dataset_size']
        )
        display(Markdown('### Legacy low-data checkpoint comparison'))
        display(l16_200k_vs_300k)

        if not continuation_fidelity_df.empty:
            colors = plt.cm.viridis(np.linspace(0.08, 0.90, len(CONTINUE_TAGS)))
            fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.2), constrained_layout=True)
            panel_specs = [
                ('PCA', 'gen_gl_q95', 'PCA q95 novelty', 'higher means farther from training images'),
                ('SSCD', 'gen_gl_q95', 'SSCD q95 novelty', 'higher means farther from training images'),
                (None, 'hist_l1', 'One-point PDF error', 'lower is better'),
                (None, 'pk_log10_mae', r'Power-spectrum error', 'lower is better'),
            ]
            for ax, (feature, metric, title, subtitle) in zip(axes.ravel(), panel_specs):
                for color, tag in zip(colors, CONTINUE_TAGS):
                    if feature is None:
                        source_df = continuation_fidelity_df
                    else:
                        source_df = continuation_novelty_df[
                            continuation_novelty_df['feature'] == feature
                        ]
                    sub = source_df[source_df['dataset_tag'] == tag].sort_values('updates_k')
                    if sub.empty or metric not in sub.columns:
                        continue
                    ax.plot(
                        sub['updates_k'],
                        sub[metric],
                        color=color,
                        marker='o',
                        ms=6.5,
                        lw=2.5,
                        label=dataset_size_label(int(sub['dataset_size'].iloc[0])),
                    )
                if metric == 'gen_gl_q95':
                    ax.axhline(0.5, color='0.35', ls=':', lw=1.3)
                    ax.set_ylim(-0.04, 1.04)
                ax.set_title(title, fontsize=18, pad=16)
                ax.text(
                    0.5,
                    1.01,
                    subtitle,
                    transform=ax.transAxes,
                    ha='center',
                    va='bottom',
                    fontsize=11.5,
                    color='0.35',
                )
                ax.set_xlabel('Optimizer updates (thousands)')
                ax.set_xticks([updates for updates, _ in CONTINUE_CHECKPOINTS])
                ax.grid(axis='y', alpha=0.16)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
            axes[0, 0].set_ylabel('q95 novelty score')
            axes[1, 0].set_ylabel(r'$L_1$ distance')
            axes[1, 1].set_ylabel(r'mean $|\log_{10}(P_{gen}/P_{real})|$')
            handles, labels = axes[0, 0].get_legend_handles_labels()
            fig.legend(
                handles,
                labels,
                title=r'Training images $N_{2D}$',
                loc='upper center',
                bbox_to_anchor=(0.5, 0.93),
                ncol=len(labels),
                frameon=False,
            )
            fig.suptitle(
                'Low-data DiT-L16 checkpoint diagnostic (not a scaling curve)',
                fontsize=24,
                y=1.01,
            )
            fig.text(
                0.5,
                0.955,
                'Legacy files reset optimizer and scheduler state at each 25k stage; '
                'use this figure to diagnose failures, not longer-training behavior.',
                ha='center',
                fontsize=12.5,
                color='#8B1A1A',
            )
            out = QUICKCHECK_DIR / 'nf_generalize_fig2_dit_l16_legacy_checkpoint_audit.png'
            fig.savefig(out, bbox_inches='tight', dpi=300)
            plt.show()
            print('wrote', out)

        display(Markdown(
            '**Interpretation:** this low-data experiment cannot show a complete scaling relation. '
            'The nonmonotonic changes are also contaminated by optimizer and scheduler resets in the '
            'legacy resume loader. The complete fixed-budget depth comparison remains the 200k sweep '
            'shown above. Rerun the continuation with restored state before interpreting any shift.'
        ))
    elif 'gen_gl_q95' in continuation_novelty_df.columns:
        display(Markdown(
            '**Generalization trajectory not plotted:** found analyzed updates '
            f'`{available_novelty_updates}`. At least two analyzed checkpoints are required; '
            'check the table audit above for missing PCA/SSCD files.'
        ))


## Fresh DiT-L16 300k replacement sweep

This section uses the clean replacement experiment, not the failed staged
continuation. Ten DiT-L16 models start from new seed-123 initializations and
train directly to 300k optimizer updates, one for every training-set size from
$2^6$ through $2^{15}$.

DiT-L8 and DiT-L12 use their original 200k runs. DiT-L16 uses the clean 300k
replacement sweep. This is therefore a depth comparison at unequal training
budgets, labeled explicitly in the figure. No failed continuation or old L16
table is substituted. The q95 novelty score tests proximity to training
examples; **q95 novelty does not guarantee physical fidelity**.


In [ ]:
FRESH_SWEEP_NAME = 'nf_generalize_fig2_dit_l16_fresh300k_v2'
FRESH_EXPECTED_POWERS = list(range(6, 16))
FRESH_EXPECTED_TAGS = [f'd2p{power:02d}' for power in FRESH_EXPECTED_POWERS]
FRESH_EXPECTED_SIZES = [2 ** power for power in FRESH_EXPECTED_POWERS]


def fresh_300k_v2_metric_path(feature: str) -> Path:
    return TABLE_DIR / (
        f'{FRESH_SWEEP_NAME}_{feature.lower()}_full_nn_metrics.csv'
    )


def audit_fresh_300k_v2_table(feature: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    path = fresh_300k_v2_metric_path(feature)
    audit_row: dict[str, Any] = {
        'feature': feature,
        'path': rel(path),
        'exists': path.exists(),
        'rows': 0,
        'missing_tags': list(FRESH_EXPECTED_TAGS),
        'duplicate_tags': [],
        'complete': False,
    }
    if not path.exists():
        return pd.DataFrame(), audit_row

    table = ensure_arch_columns(add_generalization_columns(pd.read_csv(path)))
    table = table[table['arch'].astype(str) == 'dit_l16'].copy()
    table = table.sort_values('dataset_size')
    tags = table['dataset_tag'].dropna().astype(str)
    missing = sorted(set(FRESH_EXPECTED_TAGS) - set(tags))
    extra = sorted(set(tags) - set(FRESH_EXPECTED_TAGS))
    duplicates = sorted(tags[tags.duplicated(keep=False)].unique().tolist())
    complete = (
        len(table) == 10
        and not missing
        and not extra
        and not duplicates
        and 'gen_gl_q95' in table.columns
        and table['gen_gl_q95'].notna().all()
    )
    audit_row.update({
        'rows': len(table),
        'missing_tags': missing,
        'extra_tags': extra,
        'duplicate_tags': duplicates,
        'complete': bool(complete),
    })
    return table, audit_row


fresh_300k_v2_metrics: dict[str, pd.DataFrame] = {}
fresh_300k_v2_audit_rows: list[dict[str, Any]] = []
for feature in ('PCA', 'SSCD'):
    table, audit_row = audit_fresh_300k_v2_table(feature)
    fresh_300k_v2_metrics[feature] = table
    fresh_300k_v2_audit_rows.append(audit_row)

fresh_300k_v2_audit_df = pd.DataFrame(fresh_300k_v2_audit_rows)
display(Markdown('### Clean replacement table audit'))
display(fresh_300k_v2_audit_df)

fresh_300k_v2_complete = (
    len(fresh_300k_v2_audit_df) == 2
    and bool(fresh_300k_v2_audit_df['complete'].all())
)
if fresh_300k_v2_complete:
    display(Markdown(
        '**Fresh 300k v2 audit passed.** All ten data sizes are present exactly '
        'once in both PCA and SSCD.'
    ))
else:
    display(Markdown(
        '**Fresh 300k v2 audit incomplete: not drawing the replacement L16 '
        'curve.** Both tables must contain all ten data sizes exactly once.'
    ))


In [ ]:
FRESH_DEPTH_COLORS = {
    'dit_l8': '#009E73',
    'dit_base': '#0072B2',
    'dit_l16': '#B33C86',
}
FRESH_DEPTH_MARKERS = {'dit_l8': 'P', 'dit_base': 'D', 'dit_l16': 'X'}
FRESH_DEPTH_LABELS = {
    'dit_l8': 'DiT-L8 200k',
    'dit_base': 'DiT-L12 / base 200k',
    'dit_l16': 'DiT-L16 300k',
}


def require_fresh_complete_curve(
    table: pd.DataFrame,
    arch: str,
    *,
    context: str,
) -> pd.DataFrame:
    sub = table[table['arch'].astype(str) == arch].copy()
    sub = sub.sort_values('dataset_size')
    tags = sub['dataset_tag'].dropna().astype(str)
    missing = sorted(set(FRESH_EXPECTED_TAGS) - set(tags))
    extra = sorted(set(tags) - set(FRESH_EXPECTED_TAGS))
    if (
        len(sub) != 10
        or missing
        or extra
        or tags.duplicated().any()
        or 'gen_gl_q95' not in sub.columns
        or sub['gen_gl_q95'].isna().any()
    ):
        raise ValueError(
            f'{context} is incomplete: rows={len(sub)}, '
            f'missing={missing}, extra={extra}'
        )
    return sub


def fresh_depth_sources(feature: str) -> dict[str, pd.DataFrame]:
    baseline = pca_metrics if feature == 'PCA' else sscd_metrics
    return {
        'dit_l8': require_fresh_complete_curve(
            baseline, 'dit_l8', context=f'{feature} DiT-L8 200k'
        ),
        'dit_base': require_fresh_complete_curve(
            baseline, 'dit_base', context=f'{feature} DiT-L12 200k'
        ),
        'dit_l16': require_fresh_complete_curve(
            fresh_300k_v2_metrics[feature],
            'dit_l16',
            context=f'{feature} fresh DiT-L16 300k v2',
        ),
    }


def plot_fresh_300k_v2_depth_comparison(
    *,
    output_name: str,
    zoom_max_power: int | None = None,
) -> Path:
    fig, axes = plt.subplots(
        1, 2, figsize=(16.2, 6.2), sharey=True, constrained_layout=True
    )
    for ax, feature in zip(axes, ('PCA', 'SSCD')):
        for arch, sub in fresh_depth_sources(feature).items():
            ax.plot(
                sub['dataset_size'],
                sub['gen_gl_q95'],
                color=FRESH_DEPTH_COLORS[arch],
                marker=FRESH_DEPTH_MARKERS[arch],
                ms=9,
                lw=3,
                label=FRESH_DEPTH_LABELS[arch],
            )
        ax.axhline(0.5, color='0.35', ls=':', lw=1.6)
        shown_powers = [
            power for power in FRESH_EXPECTED_POWERS
            if zoom_max_power is None or power <= zoom_max_power
        ]
        ax.set_xscale('log', base=2)
        ax.set_xticks([2 ** power for power in shown_powers])
        ax.set_xticklabels([rf'$2^{{{power}}}$' for power in shown_powers])
        if zoom_max_power is not None:
            ax.set_xlim(2 ** 5.75, 2 ** (zoom_max_power + 0.25))
        ax.set_ylim(-0.04, 1.04)
        ax.set_xlabel(r'Training images $N_{2D}$')
        ax.set_title(f'{feature} q95 novelty', fontsize=19, pad=12)
        ax.grid(axis='y', alpha=0.18)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    axes[0].set_ylabel('q95 novelty score')
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.91),
        ncol=3, frameon=False
    )
    title = 'Clean DiT depth comparison'
    if zoom_max_power is not None:
        title += ': transition region'
    fig.suptitle(title, fontsize=23, fontweight='semibold', y=1.01)
    fig.text(
        0.5, 0.945,
        'L8 and L12 use 200k updates; fresh L16 uses 300k updates.',
        ha='center', color='0.35', fontsize=12.5
    )
    out = QUICKCHECK_DIR / output_name
    fig.savefig(out, bbox_inches='tight', dpi=300)
    plt.show()
    print('wrote', out)
    return out


fresh_300k_v2_outputs = []
if fresh_300k_v2_complete:
    fresh_300k_v2_outputs.append(plot_fresh_300k_v2_depth_comparison(
        output_name='nf_generalize_fig2_dit_l16_fresh300k_v2_depth_comparison_full.png'
    ))
    fresh_300k_v2_outputs.append(plot_fresh_300k_v2_depth_comparison(
        output_name='nf_generalize_fig2_dit_l16_fresh300k_v2_depth_comparison_zoom.png',
        zoom_max_power=11,
    ))


### Interpretation

The full-range figure answers whether the clean L16 transition is monotonic
across all ten data sizes. The zoomed figure shows the transition region without
hiding the high-data runs. A rightward L16 transition would be consistent with a
larger data requirement, but the experiment measures that outcome; it does not
assume or enforce it. Read PCA and SSCD together and verify generated maps,
one-point distributions, and power spectra before treating high novelty as a
successful scientific model.


## Takeaways


In [ ]:
lines = ['### Notebook summary']
if fresh_300k_v2_complete:
    lines.append(
        '- **Fresh 300k v2 status: complete.** PCA and SSCD both contain '
        'all ten data sizes from $2^6$ through $2^{15}$.'
    )
else:
    lines.append(
        '- **Fresh 300k v2 status: incomplete.** The replacement L16 depth '
        'curve is withheld until PCA and SSCD both contain all ten data sizes.'
    )
lines.extend([
    '- The plotted L16 line comes only from the clean replacement sweep.',
    '- DiT-L8 and DiT-L12 use 200k updates; DiT-L16 uses 300k, so this is '
    'not an equal-compute comparison.',
    '- PCA and SSCD measure novelty relative to training examples. They do '
    'not establish physical validity by themselves.',
])
display(Markdown('\n'.join(lines)))


## Great Lakes Rerun Command

From the repo root on Great Lakes:

```bash
cd /home/jiamingp/diffusion_models_repo
jupyter nbconvert --execute --to notebook --inplace notebooks/nf_generalize_fig2_dit_results.ipynb
```

If you are using the Jupyter web session, just open this notebook and run all cells. The heavy PCA/SSCD nearest-neighbor work should not rerun here; this notebook reads the completed CSV and PNG outputs.
